# Turnover-Rate Interpretability

This notebook inspects the **Turnover** vectorization (topological turnover rate per frame,
for H0, H1 and H2) to check whether it follows the temporal structure of the stimuli.

For each mouse, the turnover rate is **averaged across all trials of the same class**
(mean ± SEM, computed per frame), and segment transitions from the video metadata are marked:

- **Dashed vertical lines**: transitions present at that frame in at least
  `CONSISTENT_FRAC` of the trials of the class (fixed-structure stimuli such as
  RandomDots, GaussianDot, Gabor and PinkNoise).
- **NaturalImages** (classes in `IMAGE_ONSET_LABELS`): only the image (non-background) segments
  are used. A dashed line marks the onset of the k-th image averaged across the videos shown to
  the mouse, and a dotted line marks `IMAGE_OFFSET_FRAMES` (16) frames later, i.e. the average
  first background frame after the image.
- **Grey rug at the bottom**: transitions that vary from video to video (NaturalVideo scene cuts). The rug height is proportional to the fraction of trials with
  a transition at that frame.

Turnover features are read from the precomputed caches (`compute_vectorization_cache.py`),
layout `[H0 | H1 | H2]`, each of length `clip_frames_effective`.
Turnover index `t` counts births/deaths whose endpoint equals `t`, so it is aligned with
segment `frame_start` values (possibly off by one frame, depending on the zigzag time convention).

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import numpy as np
import pandas as pd

# Lighter figures embedded in the notebook (files written with savefig keep their own dpi)
%config InlineBackend.figure_formats = ["jpeg"]
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pil_kwargs": {"quality": 80}}
plt.rcParams["figure.dpi"] = 72

# Make scripts/utils.py importable
SCRIPTS_DIR = Path("../scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import utils as utils_module
utils_module = importlib.reload(utils_module)

_short_mouse_name = utils_module._short_mouse_name
_discover_mice = utils_module._discover_mice
_build_zz_folder = utils_module._build_zz_folder
build_vectorization_cache_stem = utils_module.build_vectorization_cache_stem
load_vectorization_cache = utils_module.load_vectorization_cache
load_trial_metadata = utils_module.load_trial_metadata
load_video_metadata_json = utils_module.load_video_metadata_json
_coerce_segment_records = utils_module._coerce_segment_records
_to_int_or_none = utils_module._to_int_or_none
_normalize_video_id_token = utils_module._normalize_video_id_token

In [ ]:
# -- Configuration ------------------------------------------------------------
DATA_ROOT = Path(
    "/orfeo/scratch/area/ygardinazzi/sensorium_2026/derivatives/"
    "grid-15x15x10_norm-by_minmax"
)
META_ROOT = next(
    p for p in [
        Path("/u/mdmc/anaflom/projects_mdmc/sensorium/metadata"),
        Path("/u/area/anaflom/projects_mdmc/sensorium/metadata"),
    ] if p.exists()
)

# Vectorization cache selection
VECTORIZATION_METHOD = "Turnover"
P_ACTIVE = 70
PER_TRIAL_THRESH = True
CLIP_FRAMES = None          # None = full-trial caches (clip_frames_effective = max valid_frames)
# Folder (inside <DATA_ROOT>/<mouse>/) holding the vectorization caches.
# "old_backup_cache" = caches from the barcodes computed before the zz-top cone-dimension fix (2026-09-24);
# "cache" = caches recomputed with the fixed zz-top. Only H0 is reliable with the old caches.
TURNOVER_CACHE_SUBDIR = "cache"

DIM_NAMES = ["H0", "H1", "H2"]

# Frame 0 (all bars are born) and frame valid_frames (all bars die) are boundary
# artefacts that dwarf the rest of the curve. Frames >= valid_frames are always masked.
DROP_FIRST_FRAME = True

# A transition frame shared by at least this fraction of the class trials is drawn as a line;
# the remaining (video-specific) transitions are drawn as a rug.
CONSISTENT_FRAC = 0.5

# H1/H2 show strong onset/offset transients (bars building up / tearing down) that flatten the
# rest of the curve. y-limits are computed ignoring this many frames at each end of the trial
# (the curve is still drawn, just clipped). Set to 0 to use the full range.
Y_LIM_IGNORE_EDGE_FRAMES = 20

# Classes made of images interleaved with a background segment. For these, only the image
# segments are marked: onset of the k-th image averaged across videos (dashed), and the first
# background frame after it, IMAGE_OFFSET_FRAMES later (dotted). The background segment is the
# most frequent segment_ID within each video (s663048 for NaturalImages).
IMAGE_ONSET_LABELS = {"NaturalImages"}
IMAGE_OFFSET_FRAMES = 16

LABELS_ORDER = ["NaturalVideo", "NaturalImages", "PinkNoise", "RandomDots", "GaussianDot", "Gabor"]

# One colour per homology dimension (categorical slots 1-3, fixed order)
DIM_COLOURS = {"H0": "#2a78d6", "H1": "#eb6834", "H2": "#1baf7a"}

SAVE_FIGURES = False
FIGURES_FOLDER = Path("figures").resolve()

MICE = _discover_mice(DATA_ROOT)
print(f"Found {len(MICE)} mice:")
for m in MICE:
    print(f"  {m}")

## Loading helpers

- `load_turnover_cache`: reads the Turnover cache of one mouse and reshapes it to
  `(n_trials, n_dims, n_frames)`, masking frames outside each trial's valid range with NaN.
- `load_trial_transitions`: for each trial, looks up its `video_ID` in the trial metadata,
  loads the video JSON (`global_meta/videos/<label>-<ID>.json`) and returns the segment
  start frames (excluding frame 0), the start frames of the non-background (image) segments,
  and the video key.

In [ ]:
def load_turnover_cache(mouse_id: str) -> dict:
    # Load one mouse's Turnover cache as a (n_trials, n_dims, n_frames) array.
    cache_stem = build_vectorization_cache_stem(
        mouse_name=mouse_id,
        method=VECTORIZATION_METHOD,
        p_active=P_ACTIVE,
        per_trial_thresh=PER_TRIAL_THRESH,
        clip_frames=CLIP_FRAMES,
    )
    cache_path = DATA_ROOT / mouse_id / TURNOVER_CACHE_SUBDIR / f"{cache_stem}.npz"
    if not cache_path.exists():
        raise FileNotFoundError(f"Missing cache: {cache_path}")

    cache = load_vectorization_cache(cache_path)
    xmat = np.asarray(cache["features"], dtype=np.float64)
    n_frames = int(cache["clip_frames_effective"])
    n_dims = xmat.shape[1] // n_frames
    if n_dims * n_frames != xmat.shape[1]:
        raise RuntimeError(f"Feature length {xmat.shape[1]} is not a multiple of n_frames={n_frames}")

    turnover = xmat.reshape(xmat.shape[0], n_dims, n_frames).copy()
    valid_frames = np.asarray(cache["valid_frames"]).astype(int)

    # Mask frames beyond each trial's valid length (includes the end-of-trial death spike)
    frame_idx = np.arange(n_frames)
    invalid = frame_idx[None, :] >= valid_frames[:, None]
    if DROP_FIRST_FRAME:
        invalid[:, 0] = True
    turnover[np.broadcast_to(invalid[:, None, :], turnover.shape)] = np.nan

    return {
        "turnover": turnover,
        "labels": np.asarray(cache["labels"]).astype(str),
        "trial_ids": np.asarray(cache["trial_ids"]).astype(int),
        "valid_frames": valid_frames,
        "n_frames": n_frames,
        "cache_path": cache_path,
    }


_video_cache = {}


def _parse_video_segments(payload):
    # Return (segment starts excluding 0, start frames of non-background segments).
    seg_records = _coerce_segment_records(payload.get("segments"))
    seg_ids = [str(seg.get("segment_id")) for seg in seg_records]
    background_id = pd.Series(seg_ids).mode().iloc[0] if seg_ids else None

    starts, image_starts = [], []
    for seg, seg_id in zip(seg_records, seg_ids):
        start = _to_int_or_none(seg.get("frame_start"))
        if start is None:
            continue
        if start > 0:
            starts.append(start)
        if seg.get("segment_id") is not None and seg_id != background_id:
            image_starts.append(start)
    return np.array(sorted(starts), dtype=int), np.array(sorted(image_starts), dtype=int)


def load_trial_transitions(mouse_id: str, trial_ids, labels):
    # Return, per trial: segment start frames (excluding 0), image-segment start frames, video key.
    df_meta = load_trial_metadata(META_ROOT, mouse_id)
    trial_to_video = dict(zip(df_meta["trial"].astype(int), df_meta["video_ID"]))

    transitions, image_starts, video_keys = [], [], []
    for trial_id, label in zip(trial_ids, labels):
        video_id = trial_to_video.get(int(trial_id))
        key = (str(label), _normalize_video_id_token(video_id))
        if key not in _video_cache:
            payload, _, _ = load_video_metadata_json(META_ROOT, video_id, label=label)
            if payload is None:
                _video_cache[key] = (False, np.array([], dtype=int), np.array([], dtype=int))
            else:
                _video_cache[key] = (True, *_parse_video_segments(payload))
        found, starts, img_starts = _video_cache[key]
        if not found:
            print(f"[{_short_mouse_name(mouse_id)}] Warning: no video JSON for trial {trial_id} ({label}, {video_id})")
        transitions.append(starts)
        image_starts.append(img_starts)
        video_keys.append(key)
    return transitions, image_starts, video_keys

## Per-class averages

For each mouse and class: mean and SEM of the turnover rate at every frame (NaN-aware, so trials
shorter than the longest one only contribute where they are valid), plus the fraction of trials
that have a segment transition at each frame.

In [ ]:
def _ordered_labels(present):
    present = list(dict.fromkeys(present))
    return [l for l in LABELS_ORDER if l in present] + sorted(l for l in present if l not in LABELS_ORDER)


def summarise_mouse(mouse_id: str) -> dict:
    data = load_turnover_cache(mouse_id)
    transitions, image_starts, video_keys = load_trial_transitions(mouse_id, data["trial_ids"], data["labels"])
    n_frames = data["n_frames"]

    per_class = {}
    for label in _ordered_labels(data["labels"]):
        sel = np.flatnonzero(data["labels"] == label)
        x = data["turnover"][sel]                                  # (n_trials, n_dims, n_frames)
        n_valid = np.sum(np.isfinite(x[:, 0, :]), axis=0)          # trials contributing per frame
        mean = np.nanmean(x, axis=0)
        sem = np.nanstd(x, axis=0, ddof=1) / np.sqrt(np.maximum(n_valid, 1))

        trans_count = np.zeros(n_frames)
        for i in sel:
            t = transitions[i]
            trans_count[t[t < n_frames]] += 1
        trans_frac = trans_count / len(sel)

        # Image classes: average onset of the k-th image across the (unique) videos of the class
        image_onsets = image_offsets = None
        n_videos = len({video_keys[i] for i in sel})
        if label in IMAGE_ONSET_LABELS:
            per_video = {video_keys[i]: image_starts[i] for i in sel if image_starts[i].size}
            n_img = {v.size for v in per_video.values()}
            if len(n_img) != 1:
                raise RuntimeError(f"{label}: videos have different numbers of images: {sorted(n_img)}")
            image_onsets = np.vstack(list(per_video.values())).mean(axis=0)
            image_offsets = image_onsets + IMAGE_OFFSET_FRAMES
            trans_frac = np.zeros(n_frames)  # replaced by the averaged image lines

        per_class[label] = {
            "n_trials": len(sel),
            "max_valid_frames": int(data["valid_frames"][sel].max()),
            "mean": mean,
            "sem": sem,
            "n_valid": n_valid,
            "trans_frac": trans_frac,
            "n_trials_with_transitions": int(sum(len(transitions[i]) > 0 for i in sel)),
            "n_videos": n_videos,
            "image_onsets": image_onsets,
            "image_offsets": image_offsets,
        }
    return {"mouse_id": mouse_id, "n_frames": n_frames, "per_class": per_class}


import warnings

summaries = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN frames beyond valid range
    for mouse_id in MICE:
        try:
            summaries[mouse_id] = summarise_mouse(mouse_id)
        except FileNotFoundError as exc:
            print(f"[{_short_mouse_name(mouse_id)}] skipped: {exc}")

rows = []
for mouse_id, s in summaries.items():
    for label, c in s["per_class"].items():
        tf = c["trans_frac"]
        rows.append({
            "mouse": _short_mouse_name(mouse_id),
            "label": label,
            "n_trials": c["n_trials"],
            "max_valid_frames": c["max_valid_frames"],
            "trials_with_transitions": c["n_trials_with_transitions"],
            "consistent_transitions": int(np.sum(tf >= CONSISTENT_FRAC)),
            "variable_transition_frames": int(np.sum((tf > 0) & (tf < CONSISTENT_FRAC))),
            "n_videos": c["n_videos"],
            "mean_image_onsets": None if c["image_onsets"] is None else np.round(c["image_onsets"], 1).tolist(),
        })
df_summary = pd.DataFrame(rows)
display(df_summary)

## Plots: mean turnover rate per class, one figure per mouse

Rows are classes, columns are homology dimensions. The y-axis is shared within each column
so classes can be compared within a mouse.

In [ ]:
def _mark_image_segments(ax, onsets, offsets):
    # Dashed line at the mean image onset, dotted line at the mean first background frame after it.
    for t in onsets:
        ax.axvline(t, color="0.25", lw=0.9, ls="--", alpha=0.8, zorder=1)
    for t in offsets:
        ax.axvline(t, color="0.55", lw=0.9, ls=":", alpha=0.9, zorder=1)


def _mark_transitions(ax, trans_frac):
    # Dashed lines for consistent transitions, grey rug for video-specific ones.
    consistent = np.flatnonzero(trans_frac >= CONSISTENT_FRAC)
    for t in consistent:
        ax.axvline(t, color="0.35", lw=0.8, ls="--", alpha=0.7, zorder=1)

    variable = np.flatnonzero((trans_frac > 0) & (trans_frac < CONSISTENT_FRAC))
    if variable.size:
        trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
        heights = 0.15 * trans_frac[variable] / trans_frac[variable].max()
        ax.vlines(variable, 0, heights, transform=trans, color="0.45", lw=1.0, zorder=1)
    return consistent.size, variable.size


def plot_mouse_turnover(summary: dict):
    per_class = summary["per_class"]
    labels = list(per_class)
    n_dims = len(DIM_NAMES)
    mouse_name = _short_mouse_name(summary["mouse_id"])

    # Shared y-limits per dimension from interior frames only
    ylims = []
    for d in range(n_dims):
        lo, hi = np.inf, -np.inf
        for c in per_class.values():
            e = Y_LIM_IGNORE_EDGE_FRAMES
            interior = slice(e, max(e + 1, c["max_valid_frames"] - e))
            band_lo = (c["mean"][d] - c["sem"][d])[interior]
            band_hi = (c["mean"][d] + c["sem"][d])[interior]
            lo, hi = min(lo, np.nanmin(band_lo)), max(hi, np.nanmax(band_hi))
        pad = 0.08 * (hi - lo)
        ylims.append((lo - pad, hi + pad))

    fig, axes = plt.subplots(
        len(labels), n_dims, figsize=(4.2 * n_dims, 2.3 * len(labels)),
        sharex=True, sharey="col", squeeze=False,
    )
    fig.suptitle(
        f"{mouse_name} | mean turnover rate per class "
        f"(p_active={P_ACTIVE}, per_trial_thresh={PER_TRIAL_THRESH})",
        fontsize=11, fontweight="bold",
    )

    for r, label in enumerate(labels):
        c = per_class[label]
        frames = np.arange(len(c["n_valid"]))
        for d, dim in enumerate(DIM_NAMES):
            ax = axes[r, d]
            n_cons, n_var = _mark_transitions(ax, c["trans_frac"])
            if c["image_onsets"] is not None:
                _mark_image_segments(ax, c["image_onsets"], c["image_offsets"])
            mean, sem = c["mean"][d], c["sem"][d]
            ax.fill_between(frames, mean - sem, mean + sem, color=DIM_COLOURS[dim], alpha=0.25, lw=0, zorder=2)
            ax.plot(frames, mean, color=DIM_COLOURS[dim], lw=1.2, zorder=3)

            ax.set_ylim(*ylims[d])
            ax.grid(axis="y", color="0.9", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            if r == 0:
                ax.set_title(dim, fontsize=10)
            if d == 0:
                trans_note = []
                if c["image_onsets"] is not None:
                    trans_note.append(
                        f"mean image onset (--) / +{IMAGE_OFFSET_FRAMES} fr (:) over {c['n_videos']} videos"
                    )
                if n_cons:
                    trans_note.append(f"{n_cons} fixed")
                if n_var:
                    trans_note.append(f"{c['n_trials_with_transitions']}/{c['n_trials']} trials w/ cuts")
                ax.set_ylabel(f"{label}\n(n={c['n_trials']})", fontsize=9)
                if trans_note:
                    ax.text(0.01, 0.97, "transitions: " + ", ".join(trans_note), transform=ax.transAxes,
                            fontsize=7, color="0.35", va="top")
            if r == len(labels) - 1:
                ax.set_xlabel("frame")

    fig.tight_layout()
    if SAVE_FIGURES:
        FIGURES_FOLDER.mkdir(parents=True, exist_ok=True)
        out = FIGURES_FOLDER / f"turnover_mean_per_class_{mouse_name}_p{P_ACTIVE}_pertrial{int(PER_TRIAL_THRESH)}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()


for mouse_id, summary in summaries.items():
    plot_mouse_turnover(summary)

## Trial-to-trial correlation of turnover curves: within vs between mice

For every video ID (class + `video_ID`), pairs of presentations are formed:

- **within**: two different presentations of the same video to the same mouse;
- **between**: one presentation in mouse A and one in mouse B (all trial combinations).

For each pair and each homology dimension, the Pearson correlation between the two turnover
curves is computed over the frames valid in both trials, excluding `CORR_IGNORE_EDGE_FRAMES`
frames at each end (onset/offset transients are shared by every trial and would inflate the
correlation regardless of the stimulus).

Note: among the eligible trials, only **NaturalVideo** clips are shown to more than one mouse
(the synthetic and NaturalImages stimuli of the other mice are in the hidden test sets), so the
between-mice comparison is available for NaturalVideo only.

In [ ]:
from itertools import combinations

CORR_IGNORE_EDGE_FRAMES = 20   # frames trimmed at each end before correlating
MIN_CORR_FRAMES = 30           # skip pairs with fewer common frames than this

# -- Collect per-trial curves and video keys for all mice --------------------------
trial_records = []   # one entry per trial
trial_curves = []    # (n_dims, n_frames) arrays, NaN-masked outside the valid range
for mouse_id in MICE:
    try:
        data = load_turnover_cache(mouse_id)
    except FileNotFoundError as exc:
        print(f"[{_short_mouse_name(mouse_id)}] skipped: {exc}")
        continue
    _, _, video_keys = load_trial_transitions(mouse_id, data["trial_ids"], data["labels"])
    for i, (label, video_token) in enumerate(video_keys):
        trial_records.append({
            "mouse": _short_mouse_name(mouse_id),
            "trial_id": int(data["trial_ids"][i]),
            "label": label,
            "video_id": video_token,
            "valid_frames": int(data["valid_frames"][i]),
            "row": len(trial_curves),
        })
        trial_curves.append(data["turnover"][i])
df_trials = pd.DataFrame(trial_records)
print(f"Trials: {len(df_trials)} from {df_trials['mouse'].nunique()} mice")


def _pair_correlations(curve_a, curve_b, vf_a, vf_b):
    # Pearson r per dimension over the common, edge-trimmed frames.
    start = max(CORR_IGNORE_EDGE_FRAMES, 1 if DROP_FIRST_FRAME else 0)
    stop = min(vf_a, vf_b) - CORR_IGNORE_EDGE_FRAMES
    if stop - start < MIN_CORR_FRAMES:
        return None, stop - start
    a, b = curve_a[:, start:stop], curve_b[:, start:stop]
    out = []
    for d in range(a.shape[0]):
        if np.std(a[d]) == 0 or np.std(b[d]) == 0:
            out.append(np.nan)
        else:
            out.append(float(np.corrcoef(a[d], b[d])[0, 1]))
    return out, stop - start


# -- Build the pair-level DataFrame ------------------------------------------------
pair_rows = []
for (label, video_id), grp in df_trials.groupby(["label", "video_id"], sort=False):
    if len(grp) < 2:
        continue
    for ia, ib in combinations(grp.index, 2):
        ta, tb = df_trials.loc[ia], df_trials.loc[ib]
        r, n_common = _pair_correlations(
            trial_curves[ta["row"]], trial_curves[tb["row"]], ta["valid_frames"], tb["valid_frames"]
        )
        if r is None:
            continue
        # Order the two mice alphabetically so a mouse pair always appears the same way
        if ta["mouse"] > tb["mouse"]:
            ta, tb = tb, ta
        for dim, r_val in zip(DIM_NAMES, r):
            pair_rows.append({
                "comparison": "within" if ta["mouse"] == tb["mouse"] else "between",
                "label": label,
                "video_id": video_id,
                "mouse_a": ta["mouse"],
                "mouse_b": tb["mouse"],
                "trial_a": int(ta["trial_id"]),
                "trial_b": int(tb["trial_id"]),
                "feature": dim,
                "r": r_val,
                "n_frames": int(n_common),
            })

df_pairs = pd.DataFrame(pair_rows)
df_pairs["label"] = pd.Categorical(df_pairs["label"], categories=_ordered_labels(df_pairs["label"]), ordered=True)
print(f"Pair-level rows: {len(df_pairs)} ({len(df_pairs) // len(DIM_NAMES)} trial pairs x {len(DIM_NAMES)} features)")
display(
    df_pairs[df_pairs["feature"] == DIM_NAMES[0]]
    .groupby(["comparison", "label"], observed=True)
    .agg(n_trial_pairs=("r", "size"), n_videos=("video_id", "nunique"))
)
display(df_pairs.head())

### Baseline: different videos of the same class

To separate stimulus-specific similarity from structure shared by all trials of a class (slow
drifts, class-wide rhythms), every same-video pair (A, B) gets a **matched control**: trial A is
kept and trial B is replaced by a random trial of **the same mouse as B**, same class, but a
**different video**. This is repeated `N_BASELINE_DRAWS` times per pair, and the draws are averaged.

The control therefore has exactly the same mice, class, number of pairs and trial A as the
original pair, and differs only in whether the video matches. All draws are stored in
`df_baseline_draws`; their per-pair mean is added to `df_pairs` as `r_baseline`.

In [ ]:
N_BASELINE_DRAWS = 5
rng = np.random.default_rng(0)

# Candidate pool per (mouse, class): trial-table indices and their video IDs
pool = {
    key: (grp.index.to_numpy(), grp["video_id"].to_numpy())
    for key, grp in df_trials.groupby(["mouse", "label"])
}
trial_index = {(m, t): i for i, (m, t) in enumerate(zip(df_trials["mouse"], df_trials["trial_id"]))}

df_pairs["pair_id"] = df_pairs.groupby(["mouse_a", "trial_a", "mouse_b", "trial_b"], sort=False).ngroup()
pair_table = df_pairs.drop_duplicates("pair_id")[["pair_id", "label", "video_id", "mouse_a", "trial_a", "mouse_b", "trial_b"]]

draw_rows = []
for p in pair_table.itertuples(index=False):
    ta = df_trials.loc[trial_index[(p.mouse_a, p.trial_a)]]
    idx, vids = pool[(p.mouse_b, str(p.label))]
    candidates = idx[vids != p.video_id]
    if candidates.size == 0:
        continue
    for draw, ib in enumerate(rng.choice(candidates, size=min(N_BASELINE_DRAWS, candidates.size), replace=False)):
        tb = df_trials.loc[ib]
        r, n_common = _pair_correlations(
            trial_curves[ta["row"]], trial_curves[tb["row"]], ta["valid_frames"], tb["valid_frames"]
        )
        if r is None:
            continue
        for dim, r_val in zip(DIM_NAMES, r):
            draw_rows.append({
                "pair_id": p.pair_id,
                "draw": draw,
                "trial_b_baseline": int(tb["trial_id"]),
                "video_id_baseline": tb["video_id"],
                "feature": dim,
                "r": r_val,
                "n_frames": int(n_common),
            })

df_baseline_draws = pd.DataFrame(draw_rows)
r_baseline = (
    df_baseline_draws.groupby(["pair_id", "feature"])["r"].mean().rename("r_baseline").reset_index()
)
df_pairs = df_pairs.drop(columns=["r_baseline"], errors="ignore").merge(r_baseline, on=["pair_id", "feature"], how="left")
df_pairs["r_minus_baseline"] = df_pairs["r"] - df_pairs["r_baseline"]
print(f"Baseline draws: {len(df_baseline_draws) // len(DIM_NAMES)} trial pairs")
print(f"Pairs without a baseline: {df_pairs['r_baseline'].isna().sum() // len(DIM_NAMES)}")
display(df_pairs.head())

### Aggregation per video

Trial pairs of the same video are not independent (each trial enters several pairs, and
NaturalVideo oracle clips are repeated ~10 times). Correlations are therefore first averaged per
**unit** = (comparison, class, video, mouse pair, feature), both for the same-video `r` and for
its matched `r_baseline`. Each box point is one video in one mouse (within) or one video in one
mouse pair (between).

Same-video vs baseline is tested per class and feature with a paired Wilcoxon signed-rank test
on the units (one-sided, same > baseline), Holm-corrected across all tests.

In [ ]:
from scipy.stats import wilcoxon


def holm_correction(pvals):
    # Holm-Bonferroni adjusted p-values (step-down, monotone).
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    m = len(pvals)
    adjusted = np.maximum.accumulate((m - np.arange(m)) * pvals[order])
    out = np.empty(m)
    out[order] = np.minimum(adjusted, 1.0)
    return out


unit_cols = ["comparison", "label", "video_id", "mouse_a", "mouse_b", "feature"]
df_units = (
    df_pairs.groupby(unit_cols, observed=True)
    .agg(
        r_mean=("r", "mean"),
        r_baseline_mean=("r_baseline", "mean"),
        n_trial_pairs=("r", "size"),
    )
    .reset_index()
)
df_units["r_minus_baseline"] = df_units["r_mean"] - df_units["r_baseline_mean"]

stat_rows = []
for (comparison, label, feature), grp in df_units.groupby(["comparison", "label", "feature"], observed=True):
    g = grp.dropna(subset=["r_mean", "r_baseline_mean"])
    p = wilcoxon(g["r_mean"], g["r_baseline_mean"], alternative="greater").pvalue if len(g) >= 5 else np.nan
    stat_rows.append({
        "comparison": comparison,
        "label": label,
        "feature": feature,
        "n_units": len(g),
        "r_same_video": g["r_mean"].mean(),
        "r_baseline": g["r_baseline_mean"].mean(),
        "difference": g["r_minus_baseline"].mean(),
        "difference_sem": g["r_minus_baseline"].sem(),
        "p_wilcoxon": p,
    })
df_corr_stats = pd.DataFrame(stat_rows)
ok = df_corr_stats["p_wilcoxon"].notna()
df_corr_stats.loc[ok, "p_holm"] = holm_correction(df_corr_stats.loc[ok, "p_wilcoxon"])

for comparison in ["within", "between"]:
    print(f"Average correlation {comparison.upper()} mice: same video vs baseline (different videos, same class)")
    display(
        df_corr_stats[df_corr_stats["comparison"] == comparison]
        .drop(columns="comparison")
        .set_index(["label", "feature"])
        .round({"r_same_video": 3, "r_baseline": 3, "difference": 3, "difference_sem": 3})
    )

In [ ]:
def _stars(p):
    if not np.isfinite(p):
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."


def boxplot_same_vs_baseline(df_u, df_stats, comparison, title):
    # One panel per feature; per class, same-video box (coloured) next to baseline box (grey).
    df_u = df_u[df_u["comparison"] == comparison]
    labels = list(df_u["label"].cat.remove_unused_categories().cat.categories)
    x = np.arange(len(labels))
    w = 0.36

    fig, axes = plt.subplots(
        1, len(DIM_NAMES), figsize=(max(4.5, 1.25 * len(labels) + 1.5) * len(DIM_NAMES) / 1.6, 4.4),
        sharey=True, squeeze=False,
    )
    fig.suptitle(title, fontsize=11, fontweight="bold")
    for d, dim in enumerate(DIM_NAMES):
        ax = axes[0, d]
        sub = df_u[df_u["feature"] == dim]
        for offset, col, colour, name in [
            (-w / 2, "r_mean", DIM_COLOURS[dim], "same video"),
            (w / 2, "r_baseline_mean", "0.6", "different video (baseline)"),
        ]:
            values = [sub.loc[sub["label"] == l, col].dropna().values for l in labels]
            bp = ax.boxplot(
                values, positions=x + offset, widths=w * 0.85, patch_artist=True,
                showfliers=False, medianprops={"color": "black", "lw": 1.2},
            )
            for box in bp["boxes"]:
                box.set_facecolor(colour)
                box.set_alpha(0.6)
                box.set_edgecolor(colour)
            means = [v.mean() if v.size else np.nan for v in values]
            ax.scatter(x + offset, means, marker="D", s=20, color="white", edgecolor="black", lw=0.8, zorder=4)

        # Significance of same > baseline above each class
        stats_dim = df_stats[(df_stats["comparison"] == comparison) & (df_stats["feature"] == dim)].set_index("label")
        for xi, l in zip(x, labels):
            if l in stats_dim.index:
                ax.text(xi, 1.0, _stars(stats_dim.loc[l, "p_holm"]), transform=ax.get_xaxis_transform(),
                        ha="center", va="bottom", fontsize=8, color="0.25")

        ax.axhline(0, color="0.6", lw=0.8)
        ax.set_xticks(x)
        n_units = sub.groupby("label", observed=True).size()
        ax.set_xticklabels([f"{l}\n(n={n_units.get(l, 0)})" for l in labels], fontsize=8, rotation=30, ha="right")
        ax.set_title(dim, fontsize=10, pad=14)
        ax.grid(axis="y", color="0.9", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
        if d == 0:
            ax.set_ylabel("Pearson r (mean per unit)")
        handles = [plt.Rectangle((0, 0), 1, 1, fc=DIM_COLOURS[dim], alpha=0.6),
                   plt.Rectangle((0, 0), 1, 1, fc="0.6", alpha=0.6)]
        ax.legend(handles, ["same video", "different video"], frameon=False, fontsize=8, loc="upper right")
    fig.tight_layout()
    return fig


suffix = f"p_active={P_ACTIVE}, per_trial_thresh={PER_TRIAL_THRESH}, edges trimmed={CORR_IGNORE_EDGE_FRAMES}"
for comparison, title in [
    ("within", "Within-mouse correlation of turnover curves: same video vs different video"),
    ("between", "Between-mice correlation of turnover curves: same video vs different video"),
]:
    fig = boxplot_same_vs_baseline(df_units, df_corr_stats, comparison, f"{title}\n({suffix})")
    if SAVE_FIGURES:
        FIGURES_FOLDER.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            FIGURES_FOLDER / f"turnover_corr_{comparison}_vs_baseline_p{P_ACTIVE}_pertrial{int(PER_TRIAL_THRESH)}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

## Turnover rate vs. frame-to-frame video change

For each trial, the stimulus video (`<RAW_DATA_ROOT>/<mouse>/data/videos/<trial>.npy`,
shape `(36, 64, n_frames)`, pixel range 0-255) is compared frame to frame:

- **MSE**(t) = mean squared error between frame t-1 and frame t (same as
  `ssdatam.videos.compute_videos_time_change`);
- **SSIM**(t) = structural similarity between frame t-1 and frame t, computed with
  `skimage.metrics.structural_similarity` using the Wang et al. (2004) settings
  (Gaussian weights σ = 1.5, `use_sample_covariance=False`, K1 = 0.01, K2 = 0.03, data range 255).

Both curves are assigned to frame t, the frame at which the change appears, so they are aligned
with the turnover index t. SSIM is a similarity (it decreases with change), so correlations use the
dissimilarity **1 − SSIM**. Both MSE and 1 − SSIM grow with visual change: if turnover follows visual
change, both correlations are expected to be positive. (The cache stores raw SSIM.)

For every trial and dimension, the Pearson correlation between the turnover curve (from the
neural response of that trial) and each video measure is computed over the trial's valid frames,
trimmed by `CORR_IGNORE_EDGE_FRAMES` at each end. `NEURAL_LAG_FRAMES` optionally shifts the
turnover curve back in time to account for response latency (0 = no shift).

**Cache.** The MSE/SSIM curves are stored in
`<DERIVATIVES_ROOT>/video_frame_change/video_frame_change_ssim-skimage-sigma<σ>_range<R>.npz`
(one row per mouse x trial, curves NaN-padded to a common length) with a `.json` sidecar
describing the content and parameters. `VIDEO_CHANGE_CACHE_MODE` controls its use:

- `"auto"`: load the cache if it exists, compute only the missing trials, and save if anything was added;
- `"load"`: only load; fail if trials are missing;
- `"recompute"`: recompute everything and overwrite the cache.

In [ ]:
from skimage.metrics import structural_similarity

RAW_DATA_ROOT = Path("/orfeo/scratch/area/ygardinazzi/sensorium/data/sensorium_all_2023")
VIDEO_MEASURES = ["MSE", "SSIM"]   # curves stored in the cache
# Measures correlated with turnover: both are dissimilarities (larger = more change between frames)
CORR_MEASURES = {
    "MSE": lambda curves: curves["MSE"],
    "1-SSIM": lambda curves: 1.0 - curves["SSIM"],
}
SSIM_SIGMA = 1.5
SSIM_DATA_RANGE = 255.0
NEURAL_LAG_FRAMES = 0   # turnover at frame t + lag is compared with the video change at frame t

DERIVATIVES_ROOT = Path("/orfeo/scratch/area/ygardinazzi/sensorium_2026/derivatives")
VIDEO_CHANGE_CACHE_MODE = "auto"   # "auto" | "load" | "recompute"


def frame_ssim(x, y, sigma=SSIM_SIGMA, data_range=SSIM_DATA_RANGE):
    # Mean SSIM between paired frames; x, y have shape (H, W, n) and are compared along the last axis.
    return np.array([
        structural_similarity(
            x[:, :, i], y[:, :, i],
            gaussian_weights=True, sigma=sigma, use_sample_covariance=False, data_range=data_range,
        )
        for i in range(x.shape[2])
    ])


def video_change_curves(mouse_id: str, trial_id: int, valid_frames: int) -> dict:
    # MSE and SSIM between consecutive frames, indexed by frame (NaN at frame 0 and beyond valid range).
    video = np.load(RAW_DATA_ROOT / mouse_id / "data" / "videos" / f"{int(trial_id)}.npy").astype(np.float64)
    n_frames = video.shape[2]
    vf = min(int(valid_frames), n_frames)
    prev, curr = video[:, :, : vf - 1], video[:, :, 1:vf]

    curves = {m: np.full(n_frames, np.nan) for m in VIDEO_MEASURES}
    curves["MSE"][1:vf] = np.mean((curr - prev) ** 2, axis=(0, 1))
    curves["SSIM"][1:vf] = frame_ssim(prev, curr)
    return curves


def _corr_with_video(turnover, video_curve, valid_frames):
    # Pearson r per dimension between turnover[t + lag] and video_curve[t] over trimmed valid frames.
    start = max(CORR_IGNORE_EDGE_FRAMES, 1)
    stop = int(valid_frames) - CORR_IGNORE_EDGE_FRAMES - NEURAL_LAG_FRAMES
    if stop - start < MIN_CORR_FRAMES:
        return [np.nan] * turnover.shape[0], stop - start
    v = video_curve[start:stop]
    out = []
    for d in range(turnover.shape[0]):
        t = turnover[d, start + NEURAL_LAG_FRAMES: stop + NEURAL_LAG_FRAMES]
        ok = np.isfinite(t) & np.isfinite(v)
        if ok.sum() < MIN_CORR_FRAMES or np.std(t[ok]) == 0 or np.std(v[ok]) == 0:
            out.append(np.nan)
        else:
            out.append(float(np.corrcoef(t[ok], v[ok])[0, 1]))
    return out, stop - start


# -- Cache of per-trial video change curves -------------------------------------------
import json
from datetime import datetime

import skimage

VIDEO_CHANGE_DIR = DERIVATIVES_ROOT / "video_frame_change"
VIDEO_CHANGE_STEM = f"video_frame_change_ssim-skimage-sigma{SSIM_SIGMA:g}_range{SSIM_DATA_RANGE:g}"
VIDEO_CHANGE_NPZ = VIDEO_CHANGE_DIR / f"{VIDEO_CHANGE_STEM}.npz"
VIDEO_CHANGE_JSON = VIDEO_CHANGE_DIR / f"{VIDEO_CHANGE_STEM}.json"


def load_video_change_cache(path=VIDEO_CHANGE_NPZ):
    # Return {(mouse_full_name, trial_id): {"MSE": curve, "SSIM": curve}} and a per-trial info table.
    data = np.load(path, allow_pickle=False)
    curves, info = {}, []
    for i, (mouse, trial) in enumerate(zip(data["mouse"], data["trial_id"])):
        curves[(str(mouse), int(trial))] = {m: data[m.lower()][i].astype(np.float64) for m in VIDEO_MEASURES}
        info.append({k: data[k][i].item() for k in ["mouse", "trial_id", "label", "video_id", "valid_frames"]})
    return curves, pd.DataFrame(info)


def save_video_change_cache(curves, info_df, path_npz=VIDEO_CHANGE_NPZ, path_json=VIDEO_CHANGE_JSON):
    # Save curves NaN-padded to a common length, plus a JSON sidecar with parameters and conventions.
    info_df = info_df.sort_values(["mouse", "trial_id"]).reset_index(drop=True)
    keys = list(zip(info_df["mouse"], info_df["trial_id"].astype(int)))
    n_max = max(len(curves[k]["MSE"]) for k in keys)
    arrays = {}
    for m in VIDEO_MEASURES:
        arr = np.full((len(keys), n_max), np.nan, dtype=np.float32)
        for i, k in enumerate(keys):
            arr[i, : len(curves[k][m])] = curves[k][m]
        arrays[m.lower()] = arr
    path_npz.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path_npz,
        mouse=info_df["mouse"].to_numpy(dtype=str),
        trial_id=info_df["trial_id"].astype(np.int64).to_numpy(),
        label=info_df["label"].to_numpy(dtype=str),
        video_id=info_df["video_id"].to_numpy(dtype=str),
        valid_frames=info_df["valid_frames"].astype(np.int64).to_numpy(),
        **arrays,
    )
    meta = {
        "description": "Frame-to-frame change of the stimulus video of each trial (Sensorium 2023).",
        "created": datetime.now().isoformat(timespec="seconds"),
        "created_by": "ZigZagSensorium/notebooks/turnover_intepretability.ipynb",
        "source_videos": str(RAW_DATA_ROOT / "<mouse>" / "data" / "videos" / "<trial>.npy"),
        "n_trials": len(keys),
        "n_mice": int(info_df["mouse"].nunique()),
        "arrays": {
            "mouse": "full recording name (str), one per row",
            "trial_id": "trial number within the recording (int)",
            "label": "video class from metadata (str)",
            "video_id": "global video ID from metadata (str)",
            "valid_frames": "number of valid frames of the trial (int)",
            "mse": f"(n_trials, {n_max}) float32; mse[i, t] = mean((frame_t - frame_(t-1))**2); NaN at t=0 and t >= valid_frames",
            "ssim": f"(n_trials, {n_max}) float32; ssim[i, t] = SSIM(frame_(t-1), frame_t); NaN at t=0 and t >= valid_frames",
        },
        "ssim_parameters": {
            "implementation": f"skimage.metrics.structural_similarity (scikit-image {skimage.__version__})",
            "gaussian_weights": True, "sigma": SSIM_SIGMA, "use_sample_covariance": False,
            "K1": 0.01, "K2": 0.03, "data_range": SSIM_DATA_RANGE,
            "reference": "Wang et al. 2004; mean SSIM over the map (borders cropped by skimage)",
        },
        "trials_included": "valid trials (valid_response & valid_trial) with a Turnover cache entry",
    }
    with open(path_json, "w") as fp:
        json.dump(meta, fp, indent=2)
    print(f"Saved {len(keys)} trials to {path_npz} (+ {path_json.name})")


if VIDEO_CHANGE_CACHE_MODE not in {"auto", "load", "recompute"}:
    raise ValueError(f"Unknown VIDEO_CHANGE_CACHE_MODE: {VIDEO_CHANGE_CACHE_MODE}")

mouse_full_name = {_short_mouse_name(m): m for m in MICE}
cached_curves, cached_info = {}, pd.DataFrame(columns=["mouse", "trial_id", "label", "video_id", "valid_frames"])
if VIDEO_CHANGE_CACHE_MODE != "recompute" and VIDEO_CHANGE_NPZ.exists():
    cached_curves, cached_info = load_video_change_cache()
    print(f"Loaded {len(cached_curves)} trials from {VIDEO_CHANGE_NPZ}")
elif VIDEO_CHANGE_CACHE_MODE == "load":
    raise FileNotFoundError(f"No video-change cache at {VIDEO_CHANGE_NPZ}")

needed = [(mouse_full_name[r.mouse], int(r.trial_id), r) for r in df_trials.itertuples(index=False)]
missing = [x for x in needed if (x[0], x[1]) not in cached_curves]
if missing and VIDEO_CHANGE_CACHE_MODE == "load":
    raise RuntimeError(f"{len(missing)} trials missing from the cache; use VIDEO_CHANGE_CACHE_MODE='auto'")

if missing:
    print(f"Computing video change for {len(missing)} trials ...")
    new_info = []
    for mouse_id, trial_id, rec in missing:
        cached_curves[(mouse_id, trial_id)] = video_change_curves(mouse_id, trial_id, rec.valid_frames)
        new_info.append({"mouse": mouse_id, "trial_id": trial_id, "label": rec.label,
                         "video_id": rec.video_id, "valid_frames": int(rec.valid_frames)})
    cached_info = pd.concat([cached_info, pd.DataFrame(new_info)], ignore_index=True)
    save_video_change_cache(cached_curves, cached_info)

# -- Per-trial correlations ---------------------------------------------------------
video_curves = {}   # (mouse, trial_id) -> {"MSE": curve, "SSIM": curve}, kept for later inspection
video_rows = []
for mouse_id, trial_id, rec in needed:
    curves = cached_curves[(mouse_id, trial_id)]
    video_curves[(rec.mouse, rec.trial_id)] = curves
    for measure, to_dissimilarity in CORR_MEASURES.items():
        r, n_common = _corr_with_video(trial_curves[rec.row], to_dissimilarity(curves), rec.valid_frames)
        for dim, r_val in zip(DIM_NAMES, r):
            video_rows.append({
                "mouse": rec.mouse,
                "trial_id": rec.trial_id,
                "label": rec.label,
                "video_id": rec.video_id,
                "valid_frames": rec.valid_frames,
                "measure": measure,
                "feature": dim,
                "r": r_val,
                "n_frames": int(n_common),
                "lag_frames": NEURAL_LAG_FRAMES,
            })

df_video_corr = pd.DataFrame(video_rows)
df_video_corr["label"] = pd.Categorical(
    df_video_corr["label"], categories=_ordered_labels(df_video_corr["label"]), ordered=True
)
print(f"Rows: {len(df_video_corr)} ({df_trials.shape[0]} trials x {len(CORR_MEASURES)} measures x {len(DIM_NAMES)} features)")
print(f"NaN correlations: {df_video_corr['r'].isna().sum()}")
display(df_video_corr.head())

### Average per video and mouse

Repeated presentations of the same video to the same mouse are averaged, so each point below is
one video in one mouse.

In [ ]:
df_video_corr_units = (
    df_video_corr.groupby(["mouse", "label", "video_id", "measure", "feature"], observed=True)
    .agg(r_mean=("r", "mean"), n_trials=("r", "size"))
    .reset_index()
)

summary_video_corr = (
    df_video_corr_units.groupby(["measure", "label", "feature"], observed=True)["r_mean"]
    .agg(n_units="size", mean="mean", sem="sem", median="median")
    .round(3)
    .unstack("feature")
)
display(summary_video_corr)

In [ ]:
def boxplot_video_corr(df_u, measure, title):
    # One panel per class; per mouse, one box per feature (H0/H1/H2).
    df_m = df_u[df_u["measure"] == measure]
    labels = list(df_m["label"].cat.remove_unused_categories().cat.categories)
    n_cols = 3
    n_rows = int(np.ceil(len(labels) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.0 * n_rows), sharey=True, squeeze=False)
    fig.suptitle(title, fontsize=11, fontweight="bold")
    w = 0.8 / len(DIM_NAMES)

    for k, label in enumerate(labels):
        ax = axes.flat[k]
        sub = df_m[df_m["label"] == label]
        mice = sorted(sub["mouse"].unique())
        x = np.arange(len(mice))
        for d, dim in enumerate(DIM_NAMES):
            values = [sub.loc[(sub["mouse"] == m) & (sub["feature"] == dim), "r_mean"].dropna().values for m in mice]
            pos = x + (d - (len(DIM_NAMES) - 1) / 2) * w
            bp = ax.boxplot(values, positions=pos, widths=w * 0.85, patch_artist=True,
                            showfliers=False, medianprops={"color": "black", "lw": 1.1})
            for box in bp["boxes"]:
                box.set_facecolor(DIM_COLOURS[dim])
                box.set_alpha(0.6)
                box.set_edgecolor(DIM_COLOURS[dim])
        n_videos = sub[sub["feature"] == DIM_NAMES[0]].groupby("mouse").size()
        ax.set_xticks(x)
        ax.set_xticklabels([f"{m.replace('rec-', '')}\n(n={n_videos.get(m, 0)})" for m in mice], fontsize=7, rotation=30, ha="right")
        ax.axhline(0, color="0.6", lw=0.8)
        ax.set_title(label, fontsize=10)
        ax.grid(axis="y", color="0.9", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
        if k % n_cols == 0:
            ax.set_ylabel(f"r(turnover, {measure})\nmean per video")
    for ax in list(axes.flat)[len(labels):]:
        ax.set_visible(False)

    handles = [plt.Rectangle((0, 0), 1, 1, fc=DIM_COLOURS[d], alpha=0.6) for d in DIM_NAMES]
    fig.legend(handles, DIM_NAMES, frameon=False, fontsize=9, loc="upper right")
    fig.tight_layout()
    return fig


suffix = (f"p_active={P_ACTIVE}, per_trial_thresh={PER_TRIAL_THRESH}, "
          f"edges trimmed={CORR_IGNORE_EDGE_FRAMES}, lag={NEURAL_LAG_FRAMES} frames")
for measure in CORR_MEASURES:
    fig = boxplot_video_corr(
        df_video_corr_units, measure,
        f"Correlation of turnover rate with frame-to-frame {measure} "
        f"(positive = more turnover when frames change more)\n({suffix})",
    )
    if SAVE_FIGURES:
        FIGURES_FOLDER.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURES_FOLDER / f"turnover_corr_video_{measure.replace('-', 'minus')}_p{P_ACTIVE}_pertrial{int(PER_TRIAL_THRESH)}_lag{NEURAL_LAG_FRAMES}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

## Event-triggered average of turnover at segment onsets

For every class with discrete segments (all except NaturalVideo), the turnover curves are cut
into windows aligned to the segment onsets and averaged, separately for each mouse.

- **Window length**: the most common segment length of the class (from the video metadata), so all
  windows of a class have the same length. For **NaturalImages** the window is aligned to the
  **image** onset (not the background) and lasts `15 (image) + 12 (background)` frames.
- **No spill-over**: frames of a window that reach the next segment (NaturalImages: the next
  image) are excluded (NaN), so each window only contains the response to one segment.
- **Trial edges**: segments starting before `ETA_EDGE_FRAMES` (i.e. the first segment, which
  contains the trial-onset transient) are excluded, and frames within `ETA_EDGE_FRAMES` of the
  end of the trial are masked (offset transient). Set `ETA_EDGE_FRAMES = 0` to keep everything.
- **Averaging**: windows are first averaged within each trial, then mean ± SEM across trials.
- **Mean removal** (`ETA_REMOVE_MEAN`, default True): each trial's averaged window has its own mean
  over the window subtracted (per feature) before averaging across trials. This removes level
  differences between mice and trials so that only the shape of the response is compared.

Frame index 0 of the window is the first frame of the segment (turnover at the frame where
the segment starts).

In [ ]:
ETA_LABELS = ["NaturalImages", "PinkNoise", "RandomDots", "GaussianDot", "Gabor"]
ETA_NATURALIMAGES_WINDOW = 15 + 12      # image + background frames after the image onset
ETA_EDGE_FRAMES = CORR_IGNORE_EDGE_FRAMES
ETA_REMOVE_MEAN = True                  # subtract each trial's mean over the window (per feature)


def _video_events(label, video_token, video_id_raw):
    # Return (event onsets, next-boundary for each onset) for one video.
    # NaturalImages: image onsets, bounded by the next image onset. Others: all segment starts,
    # bounded by the next segment start. The last event is bounded by +inf (valid_frames applies later).
    payload, _, _ = load_video_metadata_json(META_ROOT, video_id_raw, label=label)
    if payload is None:
        return np.array([], dtype=int), np.array([])
    seg_starts, image_starts = _parse_video_segments(payload)
    if label in IMAGE_ONSET_LABELS:
        onsets = image_starts
    else:
        onsets = np.concatenate([[0], seg_starts]).astype(int)   # _parse_video_segments drops frame 0
    onsets = np.unique(onsets)
    bounds = np.append(onsets[1:], np.inf)
    return onsets, bounds


# Window length per class: most common segment length across the class videos
segment_lengths = {}
for label in ETA_LABELS:
    if label in IMAGE_ONSET_LABELS:
        segment_lengths[label] = ETA_NATURALIMAGES_WINDOW
        continue
    lengths = []
    for f in (META_ROOT / "global_meta" / "videos").glob(f"{label}-*.json"):
        with open(f) as fp:
            seg = json.load(fp)["segments"]
        lengths += [e - s for s, e in zip(seg["frame_start"], seg["frame_end"])]
    segment_lengths[label] = int(pd.Series(lengths).mode().iloc[0])
print("Window length (frames) per class:", segment_lengths)

# Raw video_ID per (mouse, trial), needed to read the video JSON
raw_video_id = {}
for mouse_id in MICE:
    meta = load_trial_metadata(META_ROOT, mouse_id)
    for t, v in zip(meta["trial"].astype(int), meta["video_ID"]):
        raw_video_id[(_short_mouse_name(mouse_id), int(t))] = v

events_cache = {}
eta_trials = []   # one row per trial: per-trial mean window (n_dims, W) + event counts
for rec in df_trials[df_trials["label"].isin(ETA_LABELS)].itertuples(index=False):
    W = segment_lengths[rec.label]
    key = (rec.label, rec.video_id)
    if key not in events_cache:
        events_cache[key] = _video_events(rec.label, rec.video_id, raw_video_id[(rec.mouse, rec.trial_id)])
    onsets, bounds = events_cache[key]

    curve = trial_curves[rec.row]                       # (n_dims, n_frames), NaN outside valid range
    last_ok = rec.valid_frames - ETA_EDGE_FRAMES        # frames >= last_ok are masked
    windows = []
    for onset, bound in zip(onsets, bounds):
        if onset < ETA_EDGE_FRAMES or onset >= last_ok:
            continue
        frames = onset + np.arange(W)
        keep = (frames < bound) & (frames < last_ok) & (frames < curve.shape[1])
        win = np.full((curve.shape[0], W), np.nan)
        win[:, keep] = curve[:, frames[keep]]
        windows.append(win)
    if not windows:
        continue
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        trial_mean = np.nanmean(np.stack(windows), axis=0)
        if ETA_REMOVE_MEAN:
            trial_mean = trial_mean - np.nanmean(trial_mean, axis=1, keepdims=True)
    eta_trials.append({
        "mouse": rec.mouse, "trial_id": rec.trial_id, "label": rec.label, "video_id": rec.video_id,
        "n_events": len(windows), "window": trial_mean,
    })

# -- Tidy DataFrame: mean +- SEM across trials per mouse x class x feature x frame ---------
eta_rows = []
for (mouse, label), grp in pd.DataFrame(eta_trials).groupby(["mouse", "label"]):
    stack = np.stack(grp["window"].to_list())           # (n_trials, n_dims, W)
    n_valid = np.sum(np.isfinite(stack), axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(stack, axis=0)
        sem = np.nanstd(stack, axis=0, ddof=1) / np.sqrt(np.maximum(n_valid, 1))
    for d, dim in enumerate(DIM_NAMES):
        for lag in range(stack.shape[2]):
            eta_rows.append({
                "mouse": mouse, "label": label, "feature": dim, "frame": lag,
                "mean": mean[d, lag], "sem": sem[d, lag], "n_trials": int(n_valid[d, lag]),
                "n_trials_total": len(grp), "n_events_total": int(grp["n_events"].sum()),
            })
df_eta = pd.DataFrame(eta_rows)
df_eta["label"] = pd.Categorical(df_eta["label"], categories=[l for l in ETA_LABELS], ordered=True)

display(
    df_eta[(df_eta["feature"] == DIM_NAMES[0]) & (df_eta["frame"] == 0)]
    [["label", "mouse", "n_trials_total", "n_events_total"]]
    .sort_values(["label", "mouse"]).reset_index(drop=True)
)

In [ ]:
# One colour per mouse, fixed order across all classes (categorical slots, never cycled)
MOUSE_COLOURS_LIST = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#52514e"]
eta_mice = sorted(df_eta["mouse"].unique())
mouse_colours = {m: MOUSE_COLOURS_LIST[i] for i, m in enumerate(eta_mice)}


def plot_eta(df_eta):
    labels = [l for l in ETA_LABELS if l in set(df_eta["label"])]
    fig, axes = plt.subplots(len(labels), len(DIM_NAMES), figsize=(4.3 * len(DIM_NAMES), 2.6 * len(labels)),
                             squeeze=False)
    fig.suptitle(
        f"Event-triggered average of turnover at segment onset, per mouse\n"
        f"(p_active={P_ACTIVE}, per_trial_thresh={PER_TRIAL_THRESH}, edges excluded={ETA_EDGE_FRAMES} frames, "
        f"mean removed={ETA_REMOVE_MEAN}; mean ± SEM across trials)",
        fontsize=11, fontweight="bold",
    )
    for r, label in enumerate(labels):
        for d, dim in enumerate(DIM_NAMES):
            ax = axes[r, d]
            sub = df_eta[(df_eta["label"] == label) & (df_eta["feature"] == dim)]
            for mouse, g in sub.groupby("mouse"):
                g = g.sort_values("frame")
                c = mouse_colours[mouse]
                ax.fill_between(g["frame"], g["mean"] - g["sem"], g["mean"] + g["sem"], color=c, alpha=0.2, lw=0)
                ax.plot(g["frame"], g["mean"], color=c, lw=1.4, marker="o", ms=2.5,
                        label=f"{mouse.replace('rec-', '')} (n={g['n_trials_total'].iloc[0]})")
            if label in IMAGE_ONSET_LABELS:
                ax.axvline(15, color="0.4", lw=0.9, ls=":", zorder=0)
                ax.text(15, 0.98, " background", transform=ax.get_xaxis_transform(), fontsize=7,
                        color="0.4", va="top")
            if ETA_REMOVE_MEAN:
                ax.axhline(0, color="0.6", lw=0.8, zorder=0)
            ax.grid(color="0.92", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            if r == 0:
                ax.set_title(dim, fontsize=10)
            if d == 0:
                ylab = "turnover rate\n(mean removed)" if ETA_REMOVE_MEAN else "turnover rate"
                ax.set_ylabel(f"{label}\n{ylab}", fontsize=9)
                ax.legend(frameon=False, fontsize=7, loc="best")
            if r == len(labels) - 1:
                ax.set_xlabel("frames from segment onset")
    fig.tight_layout()
    return fig


fig = plot_eta(df_eta)
if SAVE_FIGURES:
    FIGURES_FOLDER.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_FOLDER / f"turnover_eta_segments_p{P_ACTIVE}_pertrial{int(PER_TRIAL_THRESH)}_demean{int(ETA_REMOVE_MEAN)}.png",
                dpi=150, bbox_inches="tight")
plt.show()

## Temporal response function (TRF): turnover from frame-to-frame video change

For each mouse x class x video measure (MSE or 1 − SSIM) x feature (H0, H1, H2), the turnover is
modelled as a lagged linear function of the frame-to-frame change of the trial's own video:

turnover(t) = b_trial + Σ_k h(k) · x(t − k),  k ∈ `TRF_LAGS` (−5 … +30 frames)

- **No alignment across videos is needed**: every trial is paired with its own stimulus curve x;
  only the kernel h is shared. NaturalVideo uses all its trials (mostly single presentations).
- **Scaling**: turnover is z-scored within each trial (`TRF_ZSCORE_RESPONSE`). MSE and 1 − SSIM are
  used **raw** (`TRF_LOG_MSE = False`, `TRF_ZSCORE_STIMULUS = False`), so kernel units are
  "turnover SD per unit of MSE (or 1 − SSIM)" and differ between measures.
- **Per-trial intercept** (`TRF_TRIAL_INTERCEPT`): the lagged design columns and the response are
  centred within each trial, so trial-to-trial differences in the mean MSE cannot be absorbed by
  the kernel. This does not rescale x.
- **Rows**: response frames from `TRF_EDGE_FRAMES` to `valid_frames − TRF_EDGE_FRAMES`; a row is used
  only if all its lagged stimulus values exist (lags never cross trial boundaries).
- **Fitting**: ridge regression; the penalty is `alpha × mean(diag(XᵀX))` (scale-free), with alpha
  chosen per feature by inner cross-validation grouped by video.
- **Evaluation**: cross-validated prediction r (Pearson, per held-out trial). Outer folds are grouped by
  video ID: leave-one-video-out for the synthetic classes and NaturalImages (6–8 videos per mouse),
  `TRF_MAX_OUTER_FOLDS` grouped folds for NaturalVideo. The model never sees the held-out video.
- **Baselines** (same fitted kernel, applied to held-out trials, `TRF_N_BASELINE_DRAWS` draws averaged):
  (a) stimulus of a **different video** of the same class and mouse; (b) the trial's own stimulus
  **circularly shifted** by at least `TRF_MIN_SHIFT` frames within its valid range.
  Beating (b) means there is a stimulus-locked response; beating (a) means it is specific to the video.
- **Kernels**: refit on all trials (alpha by grouped CV); the spread across outer folds is kept.

In [ ]:
from sklearn.model_selection import GroupKFold

TRF_LAGS = np.arange(-5, 31)            # frames; negative lags = causality check (should be ~0)
TRF_MEASURES = ["MSE", "1-SSIM"]
TRF_LOG_MSE = False                     # use log(MSE + TRF_LOG_EPS) instead of MSE
TRF_LOG_EPS = 1.0
TRF_ZSCORE_STIMULUS = False             # z-score the stimulus measure within each trial
TRF_ZSCORE_RESPONSE = True              # z-score turnover within each trial
TRF_TRIAL_INTERCEPT = True              # centre design and response within each trial
TRF_ALPHAS = np.logspace(-4, 2, 13)     # ridge strength relative to mean(diag(X'X))
TRF_MAX_OUTER_FOLDS = 10
TRF_INNER_FOLDS = 5
TRF_N_BASELINE_DRAWS = 5
TRF_MIN_SHIFT = 30
TRF_EDGE_FRAMES = CORR_IGNORE_EDGE_FRAMES
TRF_LABELS = LABELS_ORDER
trf_rng = np.random.default_rng(0)
N_LAGS = len(TRF_LAGS)


def trf_stimulus(curves, measure):
    # Stimulus curve x(t) for one trial (NaN where undefined).
    if measure == "MSE":
        x = curves["MSE"].astype(np.float64)
        if TRF_LOG_MSE:
            x = np.log(x + TRF_LOG_EPS)
    elif measure == "1-SSIM":
        x = 1.0 - curves["SSIM"].astype(np.float64)
    else:
        raise ValueError(measure)
    if TRF_ZSCORE_STIMULUS:
        ok = np.isfinite(x)
        sd = x[ok].std()
        x = (x - x[ok].mean()) / sd if sd > 0 else np.full_like(x, np.nan)
    return x


def lagged_design(x, rows):
    # (len(rows), n_lags) matrix with x[t - k]; NaN where the lagged frame does not exist.
    idx = rows[:, None] - TRF_LAGS[None, :]
    out = np.full(idx.shape, np.nan)
    ok = (idx >= 0) & (idx < len(x))
    out[ok] = x[idx[ok]]
    return out


def prepare_trial(turnover, x, valid_frames):
    # Return (X, Y, rows) for one trial, or None if it cannot be used.
    start = max(TRF_EDGE_FRAMES, 1 if DROP_FIRST_FRAME else 0)
    stop = min(int(valid_frames) - TRF_EDGE_FRAMES, turnover.shape[1])
    rows = np.arange(start, stop)
    X = lagged_design(x, rows)
    Y = turnover[:, rows].T
    ok = np.isfinite(X).all(axis=1) & np.isfinite(Y).all(axis=1)
    if ok.sum() < MIN_CORR_FRAMES:
        return None
    X, Y, rows = X[ok], Y[ok], rows[ok]
    if X[:, TRF_LAGS == 0].std() == 0 or np.any(Y.std(axis=0) == 0):
        return None
    if TRF_ZSCORE_RESPONSE:
        Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)
    if TRF_TRIAL_INTERCEPT:
        X = X - X.mean(axis=0)
        Y = Y - Y.mean(axis=0)
    else:
        X = np.hstack([X, np.ones((X.shape[0], 1))])   # global intercept column (not penalised)
    return X, Y, rows


def _ridge_solve(XtX, XtY, alpha):
    pen = np.ones(XtX.shape[0])
    if not TRF_TRIAL_INTERCEPT:
        pen[-1] = 0.0
    lam = alpha * np.mean(np.diag(XtX)[:N_LAGS])
    return np.linalg.solve(XtX + lam * np.diag(pen), XtY)


def _colwise_r(P, Y):
    # Pearson r between matching columns of P and Y.
    Pc, Yc = P - P.mean(axis=0), Y - Y.mean(axis=0)
    den = np.sqrt((Pc ** 2).sum(axis=0) * (Yc ** 2).sum(axis=0))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, (Pc * Yc).sum(axis=0) / den, np.nan)


def _mean_trial_r(trials, idx, H):
    # Mean per-trial prediction r over trials[idx], per feature.
    return np.nanmean([_colwise_r(trials[i]["X"] @ H, trials[i]["Y"]) for i in idx], axis=0)


def _choose_alpha(trials, XtX, XtY, groups, idx):
    # Alpha per feature maximising the inner grouped-CV prediction r within trials[idx].
    n_inner = min(TRF_INNER_FOLDS, len(set(groups[idx])))
    scores = np.zeros((len(TRF_ALPHAS), XtY.shape[2]))
    for tr, te in GroupKFold(n_splits=n_inner).split(idx, groups=groups[idx]):
        tr_idx, te_idx = idx[tr], idx[te]
        XtX_tr, XtY_tr = XtX[tr_idx].sum(axis=0), XtY[tr_idx].sum(axis=0)
        for a, alpha in enumerate(TRF_ALPHAS):
            scores[a] += _mean_trial_r(trials, te_idx, _ridge_solve(XtX_tr, XtY_tr, alpha)) / n_inner
    return TRF_ALPHAS[np.argmax(scores, axis=0)]


def _fit_with_alphas(XtX, XtY, alphas):
    # Kernel matrix (p, n_dims), each feature with its own alpha.
    return np.column_stack([_ridge_solve(XtX, XtY[:, d], alphas[d]) for d in range(XtY.shape[1])])


def _baseline_r(trial, x_other):
    # Prediction r of the trial's response using a substitute stimulus curve.
    Xb = lagged_design(x_other, trial["rows"])
    ok = np.isfinite(Xb).all(axis=1)
    if ok.sum() < MIN_CORR_FRAMES:
        return np.full(trial["Y"].shape[1], np.nan)
    if not TRF_TRIAL_INTERCEPT:
        Xb = np.hstack([Xb, np.ones((Xb.shape[0], 1))])
    return _colwise_r(Xb[ok] @ trial["H_eval"], trial["Y"][ok])


def _circular_shift(x, valid_frames):
    # Circularly shift the defined part of x by a random amount >= TRF_MIN_SHIFT.
    defined = np.flatnonzero(np.isfinite(x[: int(valid_frames)]))
    seg = x[defined]
    if len(seg) < 2 * TRF_MIN_SHIFT + 1:
        return None
    out = x.copy()
    out[defined] = np.roll(seg, trf_rng.integers(TRF_MIN_SHIFT, len(seg) - TRF_MIN_SHIFT + 1))
    return out

In [ ]:
import time

kernel_rows, score_rows, fit_info = [], [], []
t_start = time.time()
for mouse in sorted(df_trials["mouse"].unique()):
    for label in TRF_LABELS:
        recs = df_trials[(df_trials["mouse"] == mouse) & (df_trials["label"] == label)]
        if recs.empty:
            continue
        for measure in TRF_MEASURES:
            # -- Prepare trials ---------------------------------------------------------
            trials, stim = [], {}
            for rec in recs.itertuples(index=False):
                x = trf_stimulus(cached_curves[(mouse_full_name[rec.mouse], int(rec.trial_id))], measure)
                stim[int(rec.trial_id)] = (x, rec.video_id, rec.valid_frames)
                prep = prepare_trial(trial_curves[rec.row], x, rec.valid_frames)
                if prep is None:
                    continue
                X, Y, rows = prep
                trials.append({"trial_id": int(rec.trial_id), "video_id": rec.video_id, "X": X, "Y": Y,
                               "rows": rows, "valid_frames": rec.valid_frames})
            groups = np.array([t["video_id"] for t in trials])
            n_videos = len(set(groups))
            if n_videos < 3:
                print(f"[{mouse} {label} {measure}] skipped: {n_videos} videos")
                continue
            XtX = np.stack([t["X"].T @ t["X"] for t in trials])
            XtY = np.stack([t["X"].T @ t["Y"] for t in trials])
            all_idx = np.arange(len(trials))
            stim_ids = np.array(list(stim.keys()))
            stim_videos = np.array([stim[k][1] for k in stim_ids])

            # -- Outer grouped CV ---------------------------------------------------------
            n_outer = min(TRF_MAX_OUTER_FOLDS, n_videos)
            fold_kernels = []
            for fold, (tr, te) in enumerate(GroupKFold(n_splits=n_outer).split(all_idx, groups=groups)):
                alphas = _choose_alpha(trials, XtX, XtY, groups, tr)
                H = _fit_with_alphas(XtX[tr].sum(axis=0), XtY[tr].sum(axis=0), alphas)
                fold_kernels.append(H[:N_LAGS])
                for i in te:
                    t = trials[i]
                    t["H_eval"] = H
                    r_cv = _colwise_r(t["X"] @ H, t["Y"])
                    # (a) different video of the same class and mouse
                    others = stim_ids[stim_videos != t["video_id"]]
                    r_a = [_baseline_r(t, stim[o][0])
                           for o in trf_rng.choice(others, size=min(TRF_N_BASELINE_DRAWS, len(others)), replace=False)]
                    # (b) own stimulus, circularly shifted
                    x_own, _, vf_own = stim[t["trial_id"]]
                    r_b = []
                    for _ in range(TRF_N_BASELINE_DRAWS):
                        x_sh = _circular_shift(x_own, vf_own)
                        if x_sh is not None:
                            r_b.append(_baseline_r(t, x_sh))
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", category=RuntimeWarning)
                        r_a = np.nanmean(r_a, axis=0) if r_a else np.full(len(DIM_NAMES), np.nan)
                        r_b = np.nanmean(r_b, axis=0) if r_b else np.full(len(DIM_NAMES), np.nan)
                    for d, dim in enumerate(DIM_NAMES):
                        score_rows.append({
                            "mouse": mouse, "label": label, "measure": measure, "feature": dim,
                            "fold": fold, "trial_id": t["trial_id"], "video_id": t["video_id"],
                            "n_frames": t["X"].shape[0], "alpha": alphas[d],
                            "r_cv": r_cv[d], "r_diff_video": r_a[d], "r_shift": r_b[d],
                        })

            # -- Final kernel on all trials ------------------------------------------------
            alphas_full = _choose_alpha(trials, XtX, XtY, groups, all_idx)
            H_full = _fit_with_alphas(XtX.sum(axis=0), XtY.sum(axis=0), alphas_full)[:N_LAGS]
            fold_kernels = np.stack(fold_kernels)
            for d, dim in enumerate(DIM_NAMES):
                for k, lag in enumerate(TRF_LAGS):
                    kernel_rows.append({
                        "mouse": mouse, "label": label, "measure": measure, "feature": dim, "lag": int(lag),
                        "weight": H_full[k, d],
                        "weight_fold_mean": fold_kernels[:, k, d].mean(),
                        "weight_fold_std": fold_kernels[:, k, d].std(ddof=1),
                        "alpha": alphas_full[d], "n_trials": len(trials), "n_videos": n_videos,
                        "n_folds": n_outer,
                    })
            fit_info.append({"mouse": mouse, "label": label, "measure": measure, "n_trials": len(trials),
                             "n_trials_skipped": len(recs) - len(trials), "n_videos": n_videos, "n_folds": n_outer})
    print(f"[{mouse}] done ({time.time() - t_start:.0f}s)")

df_trf_kernels = pd.DataFrame(kernel_rows)
df_trf_scores = pd.DataFrame(score_rows)
for df_ in (df_trf_kernels, df_trf_scores):
    df_["label"] = pd.Categorical(df_["label"], categories=_ordered_labels(df_["label"]), ordered=True)
df_trf_fits = pd.DataFrame(fit_info)
print(f"Kernels: {len(df_trf_kernels)} rows, scores: {len(df_trf_scores)} rows")
display(df_trf_fits.groupby(["label", "measure"]).agg(
    n_mice=("mouse", "size"), trials=("n_trials", "sum"), skipped=("n_trials_skipped", "sum"),
    videos_per_mouse=("n_videos", "median"), folds=("n_folds", "median")))

### Prediction performance vs baselines

Scores are averaged per **unit** = (mouse, class, video, measure, feature), i.e. repeated presentations of
a video in a mouse are averaged. The cross-validated r is compared with each baseline by a paired,
one-sided Wilcoxon signed-rank test on the units (CV r > baseline), Holm-corrected across all tests.

In [ ]:
df_trf_units = (
    df_trf_scores.groupby(["mouse", "label", "video_id", "measure", "feature"], observed=True)
    [["r_cv", "r_diff_video", "r_shift"]].mean().reset_index()
)

trf_stat_rows = []
for (measure, label, feature), g in df_trf_units.groupby(["measure", "label", "feature"], observed=True):
    row = {"measure": measure, "label": label, "feature": feature, "n_units": len(g),
           "n_mice": g["mouse"].nunique(), "r_cv": g["r_cv"].mean(),
           "r_diff_video": g["r_diff_video"].mean(), "r_shift": g["r_shift"].mean()}
    for base in ["r_diff_video", "r_shift"]:
        gg = g.dropna(subset=["r_cv", base])
        row[f"p_vs_{base[2:]}"] = (wilcoxon(gg["r_cv"], gg[base], alternative="greater").pvalue
                                   if len(gg) >= 5 else np.nan)
    trf_stat_rows.append(row)
df_trf_stats = pd.DataFrame(trf_stat_rows)
pcols = ["p_vs_diff_video", "p_vs_shift"]
flat = df_trf_stats[pcols].to_numpy().ravel()
ok = np.isfinite(flat)
adj = np.full(flat.shape, np.nan)
adj[ok] = holm_correction(flat[ok])
df_trf_stats[[c + "_holm" for c in pcols]] = adj.reshape(-1, len(pcols))

for measure in TRF_MEASURES:
    print(f"TRF prediction performance, stimulus = {measure} (mean r per unit)")
    display(
        df_trf_stats[df_trf_stats["measure"] == measure].drop(columns=["measure"] + pcols)
        .set_index(["label", "feature"]).round(3)
    )

In [ ]:
def plot_trf_kernels(df_k, measure):
    # Rows = class, columns = feature. Thin grey: each mouse; coloured: mean ± SEM across mice.
    sub_m = df_k[df_k["measure"] == measure]
    labels = list(sub_m["label"].cat.remove_unused_categories().cat.categories)
    fig, axes = plt.subplots(len(labels), len(DIM_NAMES), figsize=(4.3 * len(DIM_NAMES), 2.5 * len(labels)),
                             squeeze=False, sharex=True)
    unit = "unit MSE" if not TRF_LOG_MSE or measure != "MSE" else "unit log(MSE)"
    unit = "SD of the stimulus" if TRF_ZSCORE_STIMULUS else (unit if measure == "MSE" else "unit of 1 − SSIM")
    fig.suptitle(
        f"TRF kernels: turnover from frame-to-frame {measure} (log={TRF_LOG_MSE and measure == 'MSE'}, "
        f"stimulus z-scored={TRF_ZSCORE_STIMULUS})\n"
        f"weights in turnover SD per {unit}; grey = individual mice, colour = mean ± SEM across mice",
        fontsize=11, fontweight="bold",
    )
    for r, label in enumerate(labels):
        for d, dim in enumerate(DIM_NAMES):
            ax = axes[r, d]
            sub = sub_m[(sub_m["label"] == label) & (sub_m["feature"] == dim)]
            piv = sub.pivot(index="lag", columns="mouse", values="weight")
            for m in piv.columns:
                ax.plot(piv.index, piv[m], color="0.7", lw=0.8, zorder=2)
            mean = piv.mean(axis=1)
            if piv.shape[1] > 1:
                sem = piv.std(axis=1, ddof=1) / np.sqrt(piv.shape[1])
                ax.fill_between(piv.index, mean - sem, mean + sem, color=DIM_COLOURS[dim], alpha=0.25, lw=0, zorder=3)
            ax.plot(piv.index, mean, color=DIM_COLOURS[dim], lw=1.6, zorder=4)
            ax.axhline(0, color="0.6", lw=0.8, zorder=1)
            ax.axvline(0, color="0.4", lw=0.8, ls="--", zorder=1)
            ax.grid(color="0.93", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3))
            if r == 0:
                ax.set_title(dim, fontsize=10)
            if d == 0:
                ax.set_ylabel(f"{label}\n(n mice={piv.shape[1]})", fontsize=9)
            if r == len(labels) - 1:
                ax.set_xlabel("lag (frames after the change)")
    fig.tight_layout()
    return fig


for measure in TRF_MEASURES:
    fig = plot_trf_kernels(df_trf_kernels, measure)
    if SAVE_FIGURES:
        FIGURES_FOLDER.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURES_FOLDER / f"trf_kernels_{measure.replace('-', 'minus')}_p{P_ACTIVE}_log{int(TRF_LOG_MSE)}"
                    f"_zx{int(TRF_ZSCORE_STIMULUS)}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
def plot_trf_scores(df_u, df_s, measure):
    # One panel per feature; per class: CV r (coloured), different-video and shifted baselines (grey).
    sub_m = df_u[df_u["measure"] == measure]
    labels = list(sub_m["label"].cat.remove_unused_categories().cat.categories)
    x = np.arange(len(labels))
    w = 0.26
    fig, axes = plt.subplots(1, len(DIM_NAMES), figsize=(5.6 * len(DIM_NAMES), 4.6), sharey=True, squeeze=False)
    fig.suptitle(f"TRF cross-validated prediction r, stimulus = {measure} (unit = video x mouse)\n"
                 f"stars: CV r > different video | CV r > circular shift (Holm)",
                 fontsize=11, fontweight="bold")
    for d, dim in enumerate(DIM_NAMES):
        ax = axes[0, d]
        sub = sub_m[sub_m["feature"] == dim]
        for offset, col, colour, name in [(-w, "r_cv", DIM_COLOURS[dim], "TRF (held-out video)"),
                                          (0, "r_diff_video", "0.55", "baseline: different video"),
                                          (w, "r_shift", "0.82", "baseline: circular shift")]:
            values = [sub.loc[sub["label"] == l, col].dropna().values for l in labels]
            bp = ax.boxplot(values, positions=x + offset, widths=w * 0.85, patch_artist=True,
                            showfliers=False, medianprops={"color": "black", "lw": 1.1})
            for box in bp["boxes"]:
                box.set_facecolor(colour)
                box.set_alpha(0.7)
                box.set_edgecolor(colour)
            ax.scatter(x + offset, [v.mean() if v.size else np.nan for v in values], marker="D", s=16,
                       color="white", edgecolor="black", lw=0.7, zorder=4)
        st = df_s[(df_s["measure"] == measure) & (df_s["feature"] == dim)].set_index("label")
        for xi, l in zip(x, labels):
            if l in st.index:
                ax.text(xi, 1.0, f"{_stars(st.loc[l, 'p_vs_diff_video_holm'])} | {_stars(st.loc[l, 'p_vs_shift_holm'])}",
                        transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=7, color="0.25")
        n_units = sub.groupby("label", observed=True).size()
        ax.set_xticks(x)
        ax.set_xticklabels([f"{l}\n(n={n_units.get(l, 0)})" for l in labels], fontsize=8, rotation=30, ha="right")
        ax.axhline(0, color="0.6", lw=0.8)
        ax.set_title(dim, fontsize=10, pad=14)
        ax.grid(axis="y", color="0.92", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
        if d == 0:
            ax.set_ylabel("prediction r (mean per unit)")
        handles = [plt.Rectangle((0, 0), 1, 1, fc=c, alpha=0.7) for c in [DIM_COLOURS[dim], "0.55", "0.82"]]
        ax.legend(handles, ["TRF (held-out video)", "different video", "circular shift"],
                  frameon=False, fontsize=7, loc="upper right")
    fig.tight_layout()
    return fig


for measure in TRF_MEASURES:
    fig = plot_trf_scores(df_trf_units, df_trf_stats, measure)
    if SAVE_FIGURES:
        fig.savefig(FIGURES_FOLDER / f"trf_scores_{measure.replace('-', 'minus')}_p{P_ACTIVE}_log{int(TRF_LOG_MSE)}"
                    f"_zx{int(TRF_ZSCORE_STIMULUS)}.png", dpi=150, bbox_inches="tight")
    plt.show()

## Multi-feature TRF: which aspects of the stimulus drive each homology dimension

The single-measure TRF is extended to several stimulus regressors, each with its own lagged kernel
(`TRF_LAGS`, −5 … +30 frames), plus behavioural nuisance regressors at lag 0. All change regressors at
frame t compare frames t−1 and t.

| Regressor | Definition |
|---|---|
| `1-SSIM` | 1 − SSIM(frame t−1, frame t) |
| `structure_change` | 1 − mean SSIM *structure* term (local correlation after removing local mean and contrast) |
| `lum_on` / `lum_off` | brightening max(Δlum, 0) / darkening max(−Δlum, 0) of the global mean luminance |
| `contrast_change` | abs(Δ global RMS contrast) |
| `flow_speed` | mean optical-flow magnitude (skimage `optical_flow_ilk`) |
| `sf_low_change` / `sf_mid_change` / `sf_high_change` | abs(Δ log10 power) in spatial-frequency bands < 4, 4–12, ≥ 12 cycles per frame width |
| `onset` | impulse at segment onsets: NaturalImages image onsets; NaturalVideo scene cuts (`MF_NV_CUT_SOURCE`: **TransNetV2** by default, or the metadata cuts, which are derived from frame-to-frame MSE and miss cuts in dark scenes); other classes segment starts. Frame 0 and the last frame excluded. |
| `offset` | impulse at NaturalImages image offsets (first background frame); zero elsewhere |
| `running`, `pupil` | locomotion speed and pupil size, **separate groups**; lag 0 only (`running@0`, `pupil@0`) in variants A–D, lagged kernels over `BEH_LAGS` (−30 … +30 frames) in variant E |
| `running_accel` | running[t] − running[t−1] (signed: speeding up vs slowing down); variants E, F, lagged over `BEH_LAGS` |
| `pupil_rate` | frame-to-frame change of the pupil trace after Gaussian smoothing (σ = `PUPIL_RATE_SIGMA` frames): > 0 dilating, < 0 constricting; variant F, lagged over `BEH_LAGS` |
| `local_lum_change`, `local_contrast_change` | mean abs change of local (Gaussian-window) luminance / contrast: **computed and analysed for overlap, not used in the models** |

Per-frame features come from `scripts/compute_trial_features.py` (v2 cache in `<DERIVATIVES_ROOT>/trial_features`);
NaturalVideo scene cuts from `scripts/detect_scene_cuts.py` (`<DERIVATIVES_ROOT>/scene_cuts`).

The fits run in parallel over mice (`MF_N_JOBS` processes, default: the CPUs available to the job).

**Regressor sets (variants)**:

- **A** `1-SSIM` + lum_on/off + contrast + flow + onset/offset (previous model)
- **B** = A without `1-SSIM` (the drop-one model of A)
- **C** = A with `structure_change` instead of `1-SSIM`
- **D** = C + the three spatial-frequency change regressors (behaviour at lag 0)
- **E** = D with **lagged behaviour kernels** (running, running acceleration, pupil over −30 … +30 frames) instead of
  lag-0 behaviour weights. Behaviour lags are two-sided because the relation between behaviour and neural activity
  can go either way.
- **F** = E with the pupil kernel replaced by **pupil size at lag 0** (slow arousal level) plus a **`pupil_rate`
  kernel** (dilation / constriction dynamics) (**main variant**: full unique-contribution analysis, generalisation
  tests and kernels). In E the free pupil kernel approximated a derivative filter with strongly alternating weights;
  F states the derivative explicitly.

Rows (frames) are kept only if every lagged value of every regressor exists, so the +30-frame behaviour lags shorten
the usable part of each trial by about 11 frames at the end, for all variants alike.

**Scaling, models and evaluation** as before: turnover z-scored within trial; regressors z-scored per mouse × class
(`per_class`) or per mouse across classes (`pooled`, class-balanced; `leave_class_out`; `NV_trained`); per-trial
intercept; ridge with alpha by inner CV grouped by video; outer folds grouped by video. Unique contribution of a group
= cross-validated r(full) − r(without the group), per held-out trial; only reported where the regressor is non-zero.

In [ ]:
import os

from scipy.ndimage import gaussian_filter1d

TRIAL_FEATURES_NPZ = DERIVATIVES_ROOT / "trial_features" / "trial_features-v2_flow-ilk-r4-w10_local-s1.5_sf-4-12.npz"
MF_FROM_CACHE = ["lum_on", "lum_off", "contrast_change", "flow_speed", "structure_change", "local_lum_change",
                 "local_contrast_change", "sf_low_change", "sf_mid_change", "sf_high_change", "running", "pupil"]
BEH_LAGS = np.arange(-30, 31)                    # frames, two-sided
MF_BEHAVIOUR = ["running", "running_accel", "pupil", "pupil_rate"]
PUPIL_RATE_SIGMA = 3.0                           # frames, Gaussian smoothing before differentiating
_STIM_D = ["structure_change", "lum_on", "lum_off", "contrast_change", "flow_speed",
           "sf_low_change", "sf_mid_change", "sf_high_change", "onset", "offset"]
_BEH_LAG0 = ["running@0", "pupil@0"]              # "<regressor>@0" = only the lag-0 column of its kernel
MF_VARIANTS = {
    "A_1-SSIM": ["1-SSIM", "lum_on", "lum_off", "contrast_change", "flow_speed", "onset", "offset"] + _BEH_LAG0,
    "C_structure": ["structure_change", "lum_on", "lum_off", "contrast_change", "flow_speed", "onset", "offset"]
                   + _BEH_LAG0,
    "D_structure+SF": _STIM_D + _BEH_LAG0,
    "E_behaviour-kernels": _STIM_D + ["running", "running_accel", "pupil"],
    "F_pupil-rate": _STIM_D + ["running", "running_accel", "pupil@0", "pupil_rate"],
}
MF_MAIN = "F_pupil-rate"
MF_LAG0_VARIANT = "D_structure+SF"
# Groups dropped for the unique-contribution analysis, per variant ("all" = every regressor group)
MF_VARIANT_DROPS = {"A_1-SSIM": ["1-SSIM"], "C_structure": [], "D_structure+SF": _BEH_LAG0,
                    "E_behaviour-kernels": ["running", "running_accel", "pupil"], "F_pupil-rate": "all"}
MF_TRANSFER_SOURCE = "NaturalVideo"
MF_NV_CUT_SOURCE = "transnetv2"     # "transnetv2" | "adaptive" | "content" | "metadata"
SCENE_CUTS_NPZ = DERIVATIVES_ROOT / "scene_cuts" / "scene_cuts_naturalvideo_tol2.npz"
MF_N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

from matplotlib.colors import LinearSegmentedColormap

DIVERGING = LinearSegmentedColormap.from_list("blue_grey_orange", ["#2a78d6", "#ecebe8", "#eb6834"])
CLASS_COLOURS = dict(zip(TRF_LABELS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]))

if not TRIAL_FEATURES_NPZ.exists():
    raise FileNotFoundError(f"{TRIAL_FEATURES_NPZ} missing: run `sbatch scripts/compute_trial_features.sh` first")
_tf = np.load(TRIAL_FEATURES_NPZ)
trial_features = {
    (str(m), int(t)): {f: _tf[f][i].astype(np.float64) for f in MF_FROM_CACHE}
    for i, (m, t) in enumerate(zip(_tf["mouse"], _tf["trial_id"]))
}
print(f"Loaded per-frame features for {len(trial_features)} trials from {TRIAL_FEATURES_NPZ.name}")

nv_cuts = None
if MF_NV_CUT_SOURCE != "metadata":
    _sc = np.load(SCENE_CUTS_NPZ, allow_pickle=True)
    nv_cuts = {str(v): np.asarray(c, dtype=int) for v, c in zip(_sc["video_id"], _sc[f"cuts_{MF_NV_CUT_SOURCE}"])}
    print(f"NaturalVideo cuts: {MF_NV_CUT_SOURCE} ({len(nv_cuts)} clips, "
          f"{np.mean([len(c) for c in nv_cuts.values()]):.2f} cuts per clip) from {SCENE_CUTS_NPZ.name}")
else:
    print("NaturalVideo cuts: metadata")


def _event_frames(label, video_token, video_id_raw):
    # (onset frames, offset frames) of one video, from the video metadata.
    if label == "NaturalVideo" and nv_cuts is not None:
        if video_token not in nv_cuts:
            raise KeyError(f"No {MF_NV_CUT_SOURCE} cuts for NaturalVideo {video_token}")
        on = nv_cuts[video_token]
        return np.unique(on[on > 0]), np.array([], dtype=int)
    payload, _, _ = load_video_metadata_json(META_ROOT, video_id_raw, label=label)
    if payload is None:
        return np.array([], dtype=int), np.array([], dtype=int)
    segs = _coerce_segment_records(payload.get("segments"))
    starts = np.array([_to_int_or_none(s.get("frame_start")) for s in segs], dtype=float)
    ends = np.array([_to_int_or_none(s.get("frame_end")) for s in segs], dtype=float)
    if label in IMAGE_ONSET_LABELS:
        ids = [str(s.get("segment_id")) for s in segs]
        background = pd.Series(ids).mode().iloc[0]
        is_img = np.array([i != background for i in ids])
        on, off = starts[is_img], ends[is_img]
    else:
        on, off = starts, np.array([])
    on = on[np.isfinite(on) & (on > 0)].astype(int)
    off = off[np.isfinite(off) & (off > 0)].astype(int)
    return np.unique(on), np.unique(off)


_events_cache = {}


def trial_regressors(rec):
    # Dict of per-frame regressor curves for one trial (NaN at frames >= valid_frames).
    mouse_full = mouse_full_name[rec.mouse]
    feats = trial_features[(mouse_full, int(rec.trial_id))]
    n = len(feats["lum_on"])
    vf = int(rec.valid_frames)
    key = (rec.label, rec.video_id)
    if key not in _events_cache:
        _events_cache[key] = _event_frames(rec.label, rec.video_id, raw_video_id[(rec.mouse, int(rec.trial_id))])
    on, off = _events_cache[key]
    out = {"1-SSIM": 1.0 - cached_curves[(mouse_full, int(rec.trial_id))]["SSIM"][:n]}
    for name, frames in [("onset", on), ("offset", off)]:
        imp = np.zeros(n)
        imp[vf:] = np.nan
        imp[frames[frames < vf - 1]] = 1.0          # cuts on the last frame are edge artefacts
        out[name] = imp
    out.update(feats)
    out["running_accel"] = np.concatenate([[np.nan], np.diff(feats["running"])])
    pupil = feats["pupil"]
    ok = np.isfinite(pupil)
    rate = np.full_like(pupil, np.nan)
    if ok.sum() > 1:
        smooth = gaussian_filter1d(pupil[ok], PUPIL_RATE_SIGMA, mode="nearest")
        rate[np.flatnonzero(ok)[1:]] = np.diff(smooth)
    out["pupil_rate"] = rate
    return out


# Union design layout: one lagged block per regressor used by any variant; "<regressor>@0" groups are the lag-0
# column of the corresponding block, so every variant is a subset of the same design columns.
MF_LAGGED = list(dict.fromkeys(g.split("@")[0] for v in MF_VARIANTS.values() for g in v))
MF_LAGS_OF = {f: (BEH_LAGS if f in MF_BEHAVIOUR else TRF_LAGS) for f in MF_LAGGED}
MF_COLS, _c = {}, 0
for f in MF_LAGGED:
    MF_COLS[f] = np.arange(_c, _c + len(MF_LAGS_OF[f]))
    _c += len(MF_LAGS_OF[f])
for f in MF_BEHAVIOUR:
    MF_COLS[f"{f}@0"] = MF_COLS[f][MF_LAGS_OF[f] == 0]
MF_P = _c
print(f"Union design: {MF_P} columns; stimulus lags {TRF_LAGS[0]}..{TRF_LAGS[-1]}, "
      f"behaviour lags {BEH_LAGS[0]}..{BEH_LAGS[-1]}")


def group_lags(g):
    return np.array([0]) if g.endswith("@0") else MF_LAGS_OF[g]


def mf_lagged(x, rows, lags):
    # (len(rows), len(lags)) matrix with x[t - k]; NaN where the lagged frame does not exist.
    idx = rows[:, None] - lags[None, :]
    out = np.full(idx.shape, np.nan)
    ok = (idx >= 0) & (idx < len(x))
    out[ok] = x[idx[ok]]
    return out


def variant_groups(variant):
    return list(MF_VARIANTS[variant])


def variant_cols(variant):
    return np.concatenate([MF_COLS[g] for g in variant_groups(variant)])

### Overlap between regressors

Within-trial Pearson correlation between the per-frame regressors (frames 1 … valid_frames − 1), averaged over the
trials of each class. The second table asks what the **local** luminance / contrast changes add: the within-trial R²
of each local measure explained (lag 0, OLS) by the global measures, and by the global measures plus structure change.
1 − R² is the part of the local signal not captured by the regressors in the model.

In [ ]:
MF_OVERLAP_VARS = ["1-SSIM", "structure_change", "lum_on", "lum_off", "local_lum_change", "contrast_change",
                   "local_contrast_change", "flow_speed", "sf_low_change", "sf_mid_change", "sf_high_change",
                   "onset", "offset", "running", "running_accel", "pupil", "pupil_rate"]
LOCAL_R2_SETS = {
    "local_lum_change": {"global luminance": ["lum_on", "lum_off"],
                         "global lum + contrast + structure": ["lum_on", "lum_off", "contrast_change", "structure_change"]},
    "local_contrast_change": {"global contrast": ["contrast_change"],
                              "global lum + contrast + structure": ["lum_on", "lum_off", "contrast_change", "structure_change"]},
}


def _ols_r2(y, X):
    X1 = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ beta
    return 1.0 - resid.var() / y.var() if y.var() > 0 else np.nan


corr_by_class = {}
r2_rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for label in TRF_LABELS:
        recs = df_trials[df_trials["label"] == label]
        mats = []
        for rec in recs.itertuples(index=False):
            regs = trial_regressors(rec)
            vf = int(rec.valid_frames)
            M = np.column_stack([regs[v][1:vf] for v in MF_OVERLAP_VARS])
            M = M[np.isfinite(M).all(axis=1)]
            if len(M) < MIN_CORR_FRAMES:
                continue
            mats.append(pd.DataFrame(M, columns=MF_OVERLAP_VARS).corr().to_numpy())
            df_m = pd.DataFrame(M, columns=MF_OVERLAP_VARS)
            for local, sets in LOCAL_R2_SETS.items():
                for set_name, predictors in sets.items():
                    r2_rows.append({"label": label, "mouse": rec.mouse, "local_measure": local, "explained_by": set_name,
                                    "R2": _ols_r2(df_m[local].to_numpy(), df_m[predictors].to_numpy())})
        corr_by_class[label] = pd.DataFrame(np.nanmean(np.stack(mats), axis=0), index=MF_OVERLAP_VARS,
                                            columns=MF_OVERLAP_VARS)

df_local_r2 = pd.DataFrame(r2_rows)
display(df_local_r2.pivot_table(index="label", columns=["local_measure", "explained_by"], values="R2",
                                aggfunc="mean", observed=True).reindex(TRF_LABELS).round(2))

fig, axes = plt.subplots(2, 3, figsize=(17, 11), constrained_layout=True)
for ax, label in zip(axes.flat, TRF_LABELS):
    C = corr_by_class[label].to_numpy()
    im = ax.imshow(C, cmap=DIVERGING, vmin=-1, vmax=1)
    for i in range(C.shape[0]):
        for j in range(C.shape[1]):
            if i != j and np.isfinite(C[i, j]) and abs(C[i, j]) >= 0.3:
                ax.text(j, i, f"{C[i, j]:.1f}", ha="center", va="center", fontsize=6, color="0.1")
    ax.set_xticks(range(len(MF_OVERLAP_VARS)))
    ax.set_xticklabels(MF_OVERLAP_VARS, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(MF_OVERLAP_VARS)))
    ax.set_yticklabels(MF_OVERLAP_VARS, fontsize=7)
    ax.set_title(label, fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.6, label="mean within-trial r (|r| >= 0.3 annotated)")
fig.suptitle("Overlap between per-frame regressors, per class", fontsize=11, fontweight="bold")
if SAVE_FIGURES:
    FIGURES_FOLDER.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_FOLDER / "mf_regressor_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

### Pupil rate vs luminance (light reflex check)

Mean within-trial cross-correlation between `pupil_rate(t + k)` and `lum_on(t)` / `lum_off(t)`, per class.
A pupillary light reflex would appear as a **negative** correlation with `lum_on` (constriction after brightening)
and a positive one with `lum_off`, at positive lags of a few hundred ms.

In [ ]:
XC_LAGS = np.arange(-30, 31)


def _xcorr(a, b, lags):
    # corr(b[t + k], a[t]) for each lag k, over frames where both exist.
    out = np.full(len(lags), np.nan)
    for j, k in enumerate(lags):
        x, y = (a[: len(a) - k], b[k:]) if k >= 0 else (a[-k:], b[: len(b) + k])
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() > MIN_CORR_FRAMES and x[ok].std() > 0 and y[ok].std() > 0:
            out[j] = np.corrcoef(x[ok], y[ok])[0, 1]
    return out


xc = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for label in TRF_LABELS:
        acc = {"lum_on": [], "lum_off": []}
        for rec in df_trials[df_trials["label"] == label].itertuples(index=False):
            regs = trial_regressors(rec)
            vf = int(rec.valid_frames)
            for lum in acc:
                acc[lum].append(_xcorr(regs[lum][1:vf], regs["pupil_rate"][1:vf], XC_LAGS))
        xc[label] = {lum: np.nanmean(np.stack(v), axis=0) for lum, v in acc.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, lum in zip(axes, ["lum_on", "lum_off"]):
    for label in TRF_LABELS:
        ax.plot(XC_LAGS, xc[label][lum], color=CLASS_COLOURS[label], lw=1.3, label=label)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.axvline(0, color="0.4", lw=0.8, ls="--")
    ax.set_title(f"corr(pupil_rate(t + k), {lum}(t))", fontsize=10)
    ax.set_xlabel("k (frames; > 0: pupil change after the luminance change)")
    ax.grid(color="0.93", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("mean within-trial r")
axes[0].legend(frameon=False, fontsize=7)
fig.tight_layout()
plt.show()

In [ ]:
def mf_prepare_trial(rec, regs, turnover):
    # Union design (unscaled, centred within trial) and z-scored response for one trial.
    start = max(TRF_EDGE_FRAMES, 1 if DROP_FIRST_FRAME else 0)
    stop = min(int(rec.valid_frames) - TRF_EDGE_FRAMES, turnover.shape[1])
    rows = np.arange(start, stop)
    X = np.hstack([mf_lagged(regs[f], rows, MF_LAGS_OF[f]) for f in MF_LAGGED])
    Y = turnover[:, rows].T
    ok = np.isfinite(X).all(axis=1) & np.isfinite(Y).all(axis=1)
    if ok.sum() < MIN_CORR_FRAMES or np.any(Y[ok].std(axis=0) == 0):
        return None
    X, Y = X[ok], Y[ok]
    Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)
    return (X - X.mean(axis=0)).astype(np.float32), Y - Y.mean(axis=0)


def mf_feature_scale(regs_list):
    # Per-column scale 1/SD of each regressor over the valid frames of a set of trials (0 if constant).
    scale = np.zeros(MF_P)
    for f in MF_LAGGED:
        vals = np.concatenate([r[f][np.isfinite(r[f])] for r in regs_list])
        sd = vals.std() if vals.size else 0.0
        scale[MF_COLS[f]] = 1.0 / sd if sd > 0 else 0.0
    return scale


class MFData:
    # Per-mouse container: per-trial union design/response and cross-products.
    def __init__(self, trials):
        self.trials = trials
        self.X = [t["X"] for t in trials]
        self.Y = [t["Y"] for t in trials]
        self.XtX = np.stack([x.T.astype(np.float64) @ x.astype(np.float64) for x in self.X])
        self.XtY = np.stack([x.T.astype(np.float64) @ y for x, y in zip(self.X, self.Y)])
        self.label = np.array([t["label"] for t in trials])
        self.video = np.array([f"{t['label']}|{t['video_id']}" for t in trials])

    def gram(self, idx, w, cols, scale):
        s = scale[cols]
        G = np.tensordot(w[idx], self.XtX[idx], axes=1)[np.ix_(cols, cols)] * s[:, None] * s[None, :]
        b = np.tensordot(w[idx], self.XtY[idx], axes=1)[cols] * s[:, None]
        return G, b

    def trial_r(self, i, cols, scale, H):
        P = (self.X[i][:, cols] * scale[cols]) @ H
        Y = self.Y[i]
        return _colwise_r(P, np.tile(Y, (1, H.shape[1] // Y.shape[1])))


def _ridge_multi(G, b, alphas):
    lam0 = np.mean(np.diag(G))
    return np.hstack([np.linalg.solve(G + a * lam0 * np.eye(G.shape[0]), b) for a in alphas])


def mf_choose_alpha(data, idx, w, cols, scale):
    # Alpha per dim maximising the (weighted) mean held-out trial r in an inner grouped CV.
    groups = data.video[idx]
    n_inner = min(TRF_INNER_FOLDS, len(set(groups)))
    n_dims = data.Y[0].shape[1]
    score = np.zeros((len(TRF_ALPHAS), n_dims))
    for tr, te in GroupKFold(n_splits=n_inner).split(idx, groups=groups):
        G, b = data.gram(idx[tr], w, cols, scale)
        H = _ridge_multi(G, b, TRF_ALPHAS)
        r = np.array([data.trial_r(i, cols, scale, H) for i in idx[te]])
        ww = w[idx[te]] / w[idx[te]].sum()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            score += np.nansum(r * ww[:, None], axis=0).reshape(len(TRF_ALPHAS), n_dims)
    return TRF_ALPHAS[np.argmax(score, axis=0)]


def mf_fit(data, idx, w, cols, scale):
    alphas = mf_choose_alpha(data, idx, w, cols, scale)
    G, b = data.gram(idx, w, cols, scale)
    lam0 = np.mean(np.diag(G))
    H = np.column_stack([np.linalg.solve(G + alphas[d] * lam0 * np.eye(len(cols)), b[:, d])
                         for d in range(b.shape[1])])
    return H, alphas


def mf_active_cols(variant, scale, drop=None):
    cols = variant_cols(variant)
    if drop is not None:
        cols = np.setdiff1d(cols, MF_COLS[drop])
    return cols[scale[cols] > 0]


def class_balanced_weights(labels):
    counts = pd.Series(labels).value_counts()
    return np.array([len(labels) / (len(counts) * counts[l]) for l in labels])

In [ ]:
mf_score_rows, mf_kernel_rows = [], []


def _record_scores(data, variant, model, feature_set, idx, cols, scale, H, mouse):
    for i in idx:
        r = data.trial_r(i, cols, scale, H)
        t = data.trials[i]
        for d, dim in enumerate(DIM_NAMES):
            mf_score_rows.append({"mouse": mouse, "variant": variant, "model": model, "feature_set": feature_set,
                                  "label": t["label"], "video_id": t["video_id"], "trial_id": t["trial_id"],
                                  "feature": dim, "r": r[d]})


def _record_kernel(variant, model, label, mouse, cols, H, alphas):
    W = np.zeros((MF_P, H.shape[1]))
    W[cols] = H
    for g in variant_groups(variant):
        lags = group_lags(g)
        for d, dim in enumerate(DIM_NAMES):
            for k, lag in enumerate(lags):
                mf_kernel_rows.append({"mouse": mouse, "variant": variant, "model": model, "label": label,
                                       "regressor": g, "feature": dim, "lag": int(lag),
                                       "weight": W[MF_COLS[g][k], d], "alpha": alphas[d]})


def _drops(variant):
    d = MF_VARIANT_DROPS[variant]
    return variant_groups(variant) if d == "all" else d


def _cv_with_drops(data, variant, idx, w, scale, model, mouse, n_outer):
    # Outer grouped CV on trials idx: full model and the drop-one-group models of this variant.
    groups = data.video[idx]
    full_cols = mf_active_cols(variant, scale)
    for tr, te in GroupKFold(n_splits=n_outer).split(idx, groups=groups):
        tr_idx, te_idx = idx[tr], idx[te]
        for drop in [None] + _drops(variant):
            if drop is not None and len(np.intersect1d(MF_COLS[drop], full_cols)) == 0:
                continue    # group absent in this scope (e.g. offset outside NaturalImages)
            cols = mf_active_cols(variant, scale, drop)
            H, _ = mf_fit(data, tr_idx, w, cols, scale)
            _record_scores(data, variant, model, "full" if drop is None else f"minus_{drop}", te_idx, cols, scale,
                           H, mouse)


def fit_mouse(mouse):
    # All variants / models for one mouse; returns (mouse, score rows, kernel rows, summary).
    global mf_score_rows, mf_kernel_rows
    mf_score_rows, mf_kernel_rows = [], []
    t0 = time.time()
    recs = df_trials[df_trials["mouse"] == mouse]
    trials, regs_by_trial = [], []
    for rec in recs.itertuples(index=False):
        regs = trial_regressors(rec)
        prep = mf_prepare_trial(rec, regs, trial_curves[rec.row])
        if prep is None:
            continue
        trials.append({"label": rec.label, "video_id": rec.video_id, "trial_id": int(rec.trial_id),
                       "X": prep[0], "Y": prep[1]})
        regs_by_trial.append(regs)
    data = MFData(trials)
    ones = np.ones(len(trials))
    labels_here = [l for l in TRF_LABELS if l in set(data.label)]
    scales_class = {l: mf_feature_scale([regs_by_trial[i] for i in np.flatnonzero(data.label == l)])
                    for l in labels_here}
    multi = len(labels_here) > 1
    if multi:
        scale_m = mf_feature_scale(regs_by_trial)          # z-scored per mouse across classes
        w_bal = class_balanced_weights(data.label)
        all_idx = np.arange(len(trials))

    for variant in MF_VARIANTS:
        # 1. per-class models
        for label in labels_here:
            idx = np.flatnonzero(data.label == label)
            sc = scales_class[label]
            _cv_with_drops(data, variant, idx, ones, sc, "per_class", mouse,
                           min(TRF_MAX_OUTER_FOLDS, len(set(data.video[idx]))))
            cols = mf_active_cols(variant, sc)
            H, alphas = mf_fit(data, idx, ones, cols, sc)
            _record_kernel(variant, "per_class", label, mouse, cols, H, alphas)
        if not multi:
            continue
        # 2. pooled, class-balanced
        _cv_with_drops(data, variant, all_idx, w_bal, scale_m, "pooled", mouse, TRF_MAX_OUTER_FOLDS)
        cols = mf_active_cols(variant, scale_m)
        H, alphas = mf_fit(data, all_idx, w_bal, cols, scale_m)
        _record_kernel(variant, "pooled", "all", mouse, cols, H, alphas)
        if variant != MF_MAIN:
            continue
        # 3. leave-one-class-out and 4. NaturalVideo-trained transfer (main variant only)
        for label in labels_here:
            te, tr = np.flatnonzero(data.label == label), np.flatnonzero(data.label != label)
            Hl, _ = mf_fit(data, tr, w_bal, cols, scale_m)
            _record_scores(data, variant, "leave_class_out", "full", te, cols, scale_m, Hl, mouse)
        src = np.flatnonzero(data.label == MF_TRANSFER_SOURCE)
        Hn, alphas_n = mf_fit(data, src, ones, cols, scale_m)
        _record_kernel(variant, "NV_trained", MF_TRANSFER_SOURCE, mouse, cols, Hn, alphas_n)
        for label in labels_here:
            if label != MF_TRANSFER_SOURCE:
                _record_scores(data, variant, "NV_trained", "full", np.flatnonzero(data.label == label), cols,
                               scale_m, Hn, mouse)
    summary = f"{len(trials)} trials, classes={labels_here}, {time.time() - t0:.0f}s"
    return mouse, mf_score_rows, mf_kernel_rows, summary


def _fit_mouse_limited(args):
    # Worker: limit BLAS threads so that parallel mice do not oversubscribe the CPUs.
    mouse, n_threads = args
    from threadpoolctl import threadpool_limits
    with threadpool_limits(limits=n_threads):
        return fit_mouse(mouse)


mf_mice = sorted(df_trials["mouse"].unique(), key=lambda m: -(df_trials["mouse"] == m).sum())   # largest first
t_start = time.time()
outputs = []
if MF_N_JOBS > 1:
    import multiprocessing as mp
    n_proc = min(MF_N_JOBS, len(mf_mice))
    threads = max(1, MF_N_JOBS // n_proc)
    print(f"Fitting {len(mf_mice)} mice in {n_proc} processes x {threads} BLAS threads")
    with mp.get_context("fork").Pool(n_proc) as pool:
        for out in pool.imap_unordered(_fit_mouse_limited, [(m, threads) for m in mf_mice]):
            print(f"[{out[0]}] {out[3]} (elapsed {time.time() - t_start:.0f}s)")
            outputs.append(out)
else:
    for m in mf_mice:
        out = fit_mouse(m)
        print(f"[{out[0]}] {out[3]} (elapsed {time.time() - t_start:.0f}s)")
        outputs.append(out)
mf_score_rows = [row for out in outputs for row in out[1]]
mf_kernel_rows = [row for out in outputs for row in out[2]]

df_mf_scores = pd.DataFrame(mf_score_rows)
df_mf_kernels = pd.DataFrame(mf_kernel_rows)
for df_ in (df_mf_scores, df_mf_kernels):
    df_["label"] = pd.Categorical(df_["label"], categories=TRF_LABELS + ["all"], ordered=True)
print(f"Scores: {len(df_mf_scores)} rows, kernels: {len(df_mf_kernels)} rows")

### Comparing the regressor sets

Cross-validated prediction r of the full model (unit = mouse × video) for each variant; B is A without `1-SSIM`.

In [ ]:
full = df_mf_scores[df_mf_scores["feature_set"] == "full"].copy()
b = df_mf_scores[(df_mf_scores["variant"] == "A_1-SSIM") & (df_mf_scores["feature_set"] == "minus_1-SSIM")].copy()
b["variant"] = "B_no-SSIM"
df_mf_variant_units = (
    pd.concat([full, b], ignore_index=True)
    .groupby(["variant", "model", "mouse", "label", "video_id", "feature"], observed=True)["r"].mean().reset_index()
)
VARIANT_ORDER = ["A_1-SSIM", "B_no-SSIM", "C_structure", "D_structure+SF", "E_behaviour-kernels", "F_pupil-rate"]
VARIANT_COLOURS = dict(zip(VARIANT_ORDER, ["#eda100", "#e87ba4", "#008300", "#6250d6", "#2a78d6", "#eb6834"]))
for model in ["per_class", "pooled"]:
    print(f"Mean prediction r per variant, model = {model}")
    display(df_mf_variant_units[df_mf_variant_units["model"] == model]
            .pivot_table(index=["label", "feature"], columns="variant", values="r", observed=True)
            .reindex(columns=VARIANT_ORDER).round(3))

fig, axes = plt.subplots(2, len(DIM_NAMES), figsize=(5.6 * len(DIM_NAMES), 8), sharey="row", squeeze=False)
fig.suptitle("Prediction r per regressor set (mean ± SEM across units)", fontsize=11, fontweight="bold")
for r_i, model in enumerate(["per_class", "pooled"]):
    sub_m = df_mf_variant_units[df_mf_variant_units["model"] == model]
    labels = [l for l in TRF_LABELS if l in set(sub_m["label"])]
    x = np.arange(len(labels))
    w = 0.8 / len(VARIANT_ORDER)
    for d, dim in enumerate(DIM_NAMES):
        ax = axes[r_i, d]
        for k, v in enumerate(VARIANT_ORDER):
            g = sub_m[(sub_m["variant"] == v) & (sub_m["feature"] == dim)].groupby("label", observed=True)["r"]
            mean, sem = g.mean().reindex(labels), g.sem().reindex(labels)
            ax.bar(x + (k - (len(VARIANT_ORDER) - 1) / 2) * w, mean, width=w * 0.9, yerr=sem, color=VARIANT_COLOURS[v],
                   error_kw={"lw": 0.8, "capsize": 0}, label=v)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
        ax.set_title(f"{dim} | {model}", fontsize=10)
        ax.grid(axis="y", color="0.92", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
        if d == 0:
            ax.set_ylabel("prediction r")
        if r_i == 0 and d == 0:
            ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURES_FOLDER / f"mf_trf_variants_p{P_ACTIVE}.png", dpi=150, bbox_inches="tight")
plt.show()

### Unique contribution of each regressor (main variant)

Per held-out trial: Δr = r(full) − r(without the group). Averaged per unit (mouse × video), tested against 0
(one-sided Wilcoxon, Δr > 0), Holm-corrected across all tests of a model. A regressor is only reported for
classes where it is non-zero (`offset`: NaturalImages only). In the pooled model, dropping a regressor can still
change predictions in other classes indirectly (its variance gets absorbed by the shared kernels); that is a property
of the model, not a contribution.

In [ ]:
def unique_contributions(df_scores, variant, model):
    sub = df_scores[(df_scores["variant"] == variant) & (df_scores["model"] == model)]
    keys = ["mouse", "label", "video_id", "trial_id", "feature"]
    full = sub[sub["feature_set"] == "full"].set_index(keys)["r"]
    rows = []
    for fs in sorted(set(sub["feature_set"]) - {"full"}):
        red = sub[sub["feature_set"] == fs].set_index(keys)["r"]
        d = (full - red).dropna().rename("delta_r").reset_index()
        d["group"] = fs.replace("minus_", "")
        rows.append(d)
    per_trial = pd.concat(rows, ignore_index=True)
    per_trial = per_trial[~((per_trial["group"] == "offset") & ~per_trial["label"].isin(IMAGE_ONSET_LABELS))]
    units = (per_trial.groupby(["mouse", "label", "video_id", "feature", "group"], observed=True)["delta_r"]
             .mean().reset_index())
    stats = []
    for (label, feature, group), g in units.groupby(["label", "feature", "group"], observed=True):
        p = wilcoxon(g["delta_r"], alternative="greater").pvalue if len(g) >= 5 and np.any(g["delta_r"] != 0) else np.nan
        stats.append({"label": label, "feature": feature, "group": group, "n_units": len(g),
                      "delta_r": g["delta_r"].mean(), "p": p})
    stats = pd.DataFrame(stats)
    ok = stats["p"].notna()
    stats.loc[ok, "p_holm"] = holm_correction(stats.loc[ok, "p"].to_numpy())
    return units, stats


MAIN_GROUPS = variant_groups(MF_MAIN)
df_mf_unique_units, df_mf_unique = {}, {}
for model in ["per_class", "pooled"]:
    df_mf_unique_units[model], df_mf_unique[model] = unique_contributions(df_mf_scores, MF_MAIN, model)
    print(f"Unique contribution (mean delta r), variant = {MF_MAIN}, model = {model}")
    display(df_mf_unique[model].pivot_table(index=["label", "feature"], columns="group", values="delta_r",
                                            observed=True).reindex(columns=MAIN_GROUPS).round(3))

In [ ]:
def plot_unique_heatmaps(stats, model, groups):
    labels = [l for l in TRF_LABELS if l in set(stats["label"])]
    vmax = np.nanmax(np.abs(stats["delta_r"])) or 1.0
    fig, axes = plt.subplots(1, len(DIM_NAMES), figsize=(6.0 * len(DIM_NAMES), 0.55 * len(labels) + 2.8),
                             squeeze=False, constrained_layout=True)
    fig.suptitle(f"Unique contribution of each regressor to the cross-validated prediction r "
                 f"(variant = {MF_MAIN}, model = {model})\n"
                 f"Δr = r(full) − r(without the regressor); stars: Δr > 0 (Wilcoxon, Holm)",
                 fontsize=11, fontweight="bold")
    for d, dim in enumerate(DIM_NAMES):
        ax = axes[0, d]
        sub = stats[stats["feature"] == dim]
        M = sub.pivot_table(index="label", columns="group", values="delta_r", observed=True).reindex(index=labels, columns=groups)
        Pm = sub.pivot_table(index="label", columns="group", values="p_holm", observed=True).reindex(index=labels, columns=groups)
        im = ax.imshow(M.to_numpy(dtype=float), cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect="auto")
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                v, p = M.iat[i, j], Pm.iat[i, j]
                if np.isfinite(v):
                    s = _stars(p) if np.isfinite(p) else ""
                    ax.text(j, i, f"{v:.3f}\n{'' if s == 'n.s.' else s}", ha="center", va="center", fontsize=6.5,
                            color="0.1")
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels(groups, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels if d == 0 else [], fontsize=8)
        ax.set_title(dim, fontsize=10)
    fig.colorbar(im, ax=axes[0, :].tolist(), shrink=0.8, label="Δr")
    return fig


for model in ["per_class", "pooled"]:
    fig = plot_unique_heatmaps(df_mf_unique[model], model, MAIN_GROUPS)
    if SAVE_FIGURES:
        fig.savefig(FIGURES_FOLDER / f"mf_trf_unique_{model}_p{P_ACTIVE}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Behaviour: lag-0 weights (variant D) and unique contribution of behaviour as lag-0 weights vs kernels
beh = df_mf_kernels[(df_mf_kernels["variant"] == MF_LAG0_VARIANT) & (df_mf_kernels["model"] == "per_class")
                    & (df_mf_kernels["regressor"].isin(_BEH_LAG0))]
print(f"Lag-0 behaviour weights (turnover SD per regressor SD), variant {MF_LAG0_VARIANT}, per-class, mean across mice")
display(beh.pivot_table(index=["label"], columns=["regressor", "feature"], values="weight", aggfunc="mean",
                        observed=True).round(3))

beh_rows = []
for model in ["per_class", "pooled"]:
    _, st_d = unique_contributions(df_mf_scores, MF_LAG0_VARIANT, model)
    st_d = st_d.assign(model=model, form="lag 0 (variant D)")
    st_d["group"] = st_d["group"].str.replace("@0", "", regex=False)
    _, st_e = unique_contributions(df_mf_scores, "E_behaviour-kernels", model)
    st_e = st_e[st_e["group"].isin(MF_BEHAVIOUR)].assign(model=model, form="kernels (variant E)")
    st_f = df_mf_unique[model][df_mf_unique[model]["group"].isin(MF_BEHAVIOUR + ["pupil@0"])].assign(
        model=model, form="pupil@0 + pupil_rate kernel (variant F)")
    beh_rows += [st_d, st_e, st_f]
df_beh_unique = pd.concat(beh_rows, ignore_index=True)
print("Unique contribution (mean delta r) of behaviour: lag-0 weights (D) vs kernels (E) vs pupil level + rate (F)")
display(df_beh_unique.pivot_table(index=["model", "label"], columns=["feature", "group", "form"], values="delta_r",
                                  observed=True).round(3))

### Does one kernel fit all classes? (main variant)

Prediction r of the full model on the same held-out trials (unit = mouse × video), for the mice that saw several
classes: class-specific model, pooled model, pooled model that never saw the class, and the NaturalVideo-trained model
(no box for NaturalVideo itself: that would be the class-specific model).

In [ ]:
pool_mice = sorted(df_mf_scores.loc[df_mf_scores["model"] == "pooled", "mouse"].unique())
gen = df_mf_scores[(df_mf_scores["variant"] == MF_MAIN) & (df_mf_scores["feature_set"] == "full")
                   & (df_mf_scores["mouse"].isin(pool_mice))]
df_mf_gen_units = (gen.groupby(["model", "mouse", "label", "video_id", "feature"], observed=True)["r"]
                   .mean().reset_index())
MF_MODELS = ["per_class", "pooled", "leave_class_out", "NV_trained"]
display(df_mf_gen_units.pivot_table(index=["label", "feature"], columns="model", values="r", observed=True)
        .reindex(columns=MF_MODELS).round(3))


def plot_generalisation(df_u):
    labels = [l for l in TRF_LABELS if l in set(df_u["label"])]
    x = np.arange(len(labels))
    w = 0.2
    shades = {"per_class": None, "pooled": "0.40", "leave_class_out": "0.65", "NV_trained": "0.85"}
    hatches = {"per_class": "", "pooled": "", "leave_class_out": "//", "NV_trained": ".."}
    names = {"per_class": "class-specific", "pooled": "pooled", "leave_class_out": "pooled, class left out",
             "NV_trained": "trained on NaturalVideo"}
    fig, axes = plt.subplots(1, len(DIM_NAMES), figsize=(5.6 * len(DIM_NAMES), 4.6), sharey=True, squeeze=False)
    fig.suptitle(f"Multi-feature TRF ({MF_MAIN}): does one kernel generalise across classes? "
                 f"(mice: {len(pool_mice)}, unit = mouse x video)", fontsize=11, fontweight="bold")
    for d, dim in enumerate(DIM_NAMES):
        ax = axes[0, d]
        for k, model in enumerate(MF_MODELS):
            sub = df_u[(df_u["model"] == model) & (df_u["feature"] == dim)]
            values = [sub.loc[sub["label"] == l, "r"].dropna().values for l in labels]
            colour = DIM_COLOURS[dim] if shades[model] is None else shades[model]
            pos = x + (k - 1.5) * w
            keep = [i for i, v in enumerate(values) if v.size]
            bp = ax.boxplot([values[i] for i in keep], positions=pos[keep], widths=w * 0.85, patch_artist=True,
                            showfliers=False, medianprops={"color": "black", "lw": 1.0})
            for box in bp["boxes"]:
                box.set_facecolor(colour)
                box.set_alpha(0.75)
                box.set_edgecolor("0.3")
                box.set_hatch(hatches[model])
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
        ax.axhline(0, color="0.6", lw=0.8)
        ax.set_title(dim, fontsize=10)
        ax.grid(axis="y", color="0.92", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
        if d == 0:
            ax.set_ylabel("prediction r (mean per unit)")
        handles = [plt.Rectangle((0, 0), 1, 1, fc=DIM_COLOURS[dim] if shades[m] is None else shades[m], alpha=0.75,
                                 ec="0.3", hatch=hatches[m]) for m in MF_MODELS]
        ax.legend(handles, [names[m] for m in MF_MODELS], frameon=False, fontsize=7, loc="upper right")
    fig.tight_layout()
    return fig


fig = plot_generalisation(df_mf_gen_units)
if SAVE_FIGURES:
    fig.savefig(FIGURES_FOLDER / f"mf_trf_generalisation_p{P_ACTIVE}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def plot_pooled_kernels(df_k, variant=MF_MAIN, model="pooled"):
    # Rows = lagged stimulus regressor, columns = homology dimension; one line per mouse + mean.
    stim = [g for g in MF_VARIANTS[variant] if g not in MF_BEHAVIOUR and not g.endswith("@0")]
    sub_m = df_k[(df_k["variant"] == variant) & (df_k["model"] == model) & (df_k["regressor"].isin(stim))]
    fig, axes = plt.subplots(len(stim), len(DIM_NAMES), figsize=(4.3 * len(DIM_NAMES), 2.1 * len(stim)),
                             squeeze=False, sharex=True)
    fig.suptitle(f"Multi-feature TRF kernels, variant = {variant}, model = {model} (turnover SD per regressor SD)\n"
                 f"coloured thin lines = mice, black = mean across mice", fontsize=11, fontweight="bold")
    for r, reg in enumerate(stim):
        for d, dim in enumerate(DIM_NAMES):
            ax = axes[r, d]
            piv = sub_m[(sub_m["regressor"] == reg) & (sub_m["feature"] == dim)].pivot(index="lag", columns="mouse",
                                                                                     values="weight")
            piv = piv.loc[:, (piv != 0).any(axis=0)]      # mice where the regressor is absent (all-zero kernel)
            if piv.empty:
                ax.set_visible(False)
                continue
            for m in piv.columns:
                ax.plot(piv.index, piv[m], color=mouse_colours.get(m, "0.6"), lw=0.9, alpha=0.8,
                        label=m.replace("rec-", ""))
            ax.plot(piv.index, piv.mean(axis=1), color="black", lw=1.6)
            ax.axhline(0, color="0.6", lw=0.8)
            ax.axvline(0, color="0.4", lw=0.8, ls="--")
            ax.grid(color="0.93", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            if r == 0:
                ax.set_title(dim, fontsize=10)
            if d == 0:
                ax.set_ylabel(reg, fontsize=9)
            if r == 0 and d == len(DIM_NAMES) - 1:
                ax.legend(frameon=False, fontsize=7, loc="upper right")
            if r == len(stim) - 1:
                ax.set_xlabel("lag (frames after the change)")
    fig.tight_layout()
    return fig


fig = plot_pooled_kernels(df_mf_kernels)
if SAVE_FIGURES:
    fig.savefig(FIGURES_FOLDER / f"mf_trf_kernels_pooled_p{P_ACTIVE}.png", dpi=150, bbox_inches="tight")
plt.show()

### Behaviour kernels (variants E and F)

Kernels of running, running acceleration and pupil (E) or pupil rate (F) over −30 … +30 frames; for F the lag-0
pupil-size weight is given in the title. Positive lags: turnover follows the
behaviour; negative lags: turnover precedes it. Each panel also reports the lag-0 weight of variant D (the single
weight used before) and the sum of the kernel weights; for a slowly varying signal the lag-0 weight approximates the
whole kernel, so the two numbers are comparable.

1. Pooled model (5 mice with several classes): one line per mouse, black = mean.
2. Per-class model: mean kernel across mice for each class (all 10 mice contribute to NaturalVideo).

In [ ]:
def _lag0_weight(df_k, model, reg, dim, label=None):
    sub = df_k[(df_k["variant"] == MF_LAG0_VARIANT) & (df_k["model"] == model) & (df_k["regressor"] == f"{reg}@0")
               & (df_k["feature"] == dim)]
    if label is not None:
        sub = sub[sub["label"] == label]
    return sub["weight"].mean() if len(sub) else np.nan


def plot_behaviour_kernels(df_k, model, variant=MF_MAIN):
    rows = [g for g in MF_VARIANTS[variant] if g in MF_BEHAVIOUR]
    sub_m = df_k[(df_k["variant"] == variant) & (df_k["model"] == model) & (df_k["regressor"].isin(rows))]
    fig, axes = plt.subplots(len(rows), len(DIM_NAMES), figsize=(4.4 * len(DIM_NAMES), 2.6 * len(rows)),
                             squeeze=False, sharex=True)
    what = "coloured = mice, black = mean" if model == "pooled" else "one line per class (mean across mice)"
    lvl = ""
    if "pupil@0" in MF_VARIANTS[variant]:
        w = df_k[(df_k["variant"] == variant) & (df_k["model"] == model) & (df_k["regressor"] == "pupil@0")]
        lvl = "; pupil size (lag 0): " + ", ".join(f"{d} {w.loc[w['feature'] == d, 'weight'].mean():.3f}" for d in DIM_NAMES)
    fig.suptitle(f"Behaviour kernels, variant {variant}, model = {model} (turnover SD per regressor SD)\n{what}{lvl}",
                 fontsize=11, fontweight="bold")
    for r, reg in enumerate(rows):
        for d, dim in enumerate(DIM_NAMES):
            ax = axes[r, d]
            sub = sub_m[(sub_m["regressor"] == reg) & (sub_m["feature"] == dim)]
            if model == "pooled":
                piv = sub.pivot(index="lag", columns="mouse", values="weight")
                for m in piv.columns:
                    ax.plot(piv.index, piv[m], color=mouse_colours.get(m, "0.6"), lw=0.9, alpha=0.8,
                            label=m.replace("rec-", ""))
                mean = piv.mean(axis=1)
                ax.plot(mean.index, mean, color="black", lw=1.6)
                summed = piv.sum(axis=0).mean()
            else:
                piv = sub.groupby(["label", "lag"], observed=True)["weight"].mean().unstack("label")
                for l in [l for l in TRF_LABELS if l in piv.columns]:
                    ax.plot(piv.index, piv[l], color=CLASS_COLOURS[l], lw=1.2, label=l)
                summed = piv.sum(axis=0).mean()
            w0 = _lag0_weight(df_k, model, reg, dim)
            txt = f"Σ kernel = {summed:.3f}" + ("" if not np.isfinite(w0) else f"\nlag-0 model = {w0:.3f}")
            ax.text(0.02, 0.97, txt, transform=ax.transAxes, fontsize=7, va="top", color="0.25")
            ax.axhline(0, color="0.6", lw=0.8)
            ax.axvline(0, color="0.4", lw=0.8, ls="--")
            ax.grid(color="0.93", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            if r == 0:
                ax.set_title(dim, fontsize=10)
            if d == 0:
                ax.set_ylabel(reg, fontsize=9)
            if r == 0 and d == len(DIM_NAMES) - 1:
                ax.legend(frameon=False, fontsize=7, loc="upper right")
            if r == len(rows) - 1:
                ax.set_xlabel("lag (frames; > 0: turnover after the behaviour)")
    fig.tight_layout()
    return fig


for variant in ["E_behaviour-kernels", "F_pupil-rate"]:
    for model in ["pooled", "per_class"]:
        fig = plot_behaviour_kernels(df_mf_kernels, model, variant)
        if SAVE_FIGURES:
            fig.savefig(FIGURES_FOLDER / f"mf_trf_behaviour_kernels_{variant}_{model}_p{P_ACTIVE}.png", dpi=150,
                        bbox_inches="tight")
        plt.show()